# Paper #8 on Kaggle T4 x2: per-symbol CNN / ResNet-50 + GFP-GAN plate pipeline, on our data
The paper classifies single plate symbols (17 classes, 32x32 crops) with a small CNN (4 conv layers, kernels
22/16/8/4) and a frozen ImageNet ResNet-50, and reads whole plates with GFP-GAN -> Otsu -> contours -> CNN (p.2-5).
Here the same models (PyTorch port of the authors' Keras notebooks, their settings: batch 1, Adam, CNN 40 epochs
28x28, ResNet-50 30 epochs 150x150) are trained on **every human/verified token box of our recognition split**
(101 Bangla tokens: digits, মেট্রো, series letters, district words) - **GPU 0: CNN, GPU 1: ResNet-50**. Then the
paper's plate pipeline reads all 651 OCR test crops **with and without GFP-GAN** (the ablation the paper never ran),
in the paper's x order and in a two-line order, and the same CNN reads the human token boxes (perfect segmentation).
The paper's own data is checked in `paper08_original_check.ipynb`.

In [1]:
# ===== 0. Setup =====================================================================================================
# FAST = True  -> quick test (1 epoch, ~3 % of the images, a few test images): run this first in an interactive
#                 session and check every cell finishes. Then set FAST = False and use Save Version -> Save & Run All.
FAST = False
PAPER = "08"
import os, sys, json, glob, time, subprocess
WORK = os.environ.get("LPR_WORK", "/kaggle/working")
KIT = f"{WORK}/lprkit"; RUNS = f"{WORK}/runs"
os.makedirs(KIT, exist_ok=True); os.makedirs(RUNS, exist_ok=True); sys.path.insert(0, KIT)
os.environ["LPR_WORK"] = WORK
!pip -q install ultralytics==8.4.170 spandrel
import torch
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
if not os.environ.get("LPR_LOCAL_TEST"):
    assert torch.cuda.device_count() >= 2, "Session options -> Accelerator: GPU T4 x2 (and Internet: On)"
G0, G1 = (0, 1) if torch.cuda.device_count() >= 2 else (0, 0)   # one job per GPU; a single GPU runs them in turn
E = lambda full, fast=1: fast if FAST else full                   # epochs: full run vs quick test
FR = 0.03 if FAST else 1.0                                         # fraction of the training images
TMAX = ["--test-max", "40"] if FAST else []
print("=" * 100)
print("QUICK TEST (FAST = True): 1 epoch on ~3 % of the images. Each job ends after about 1-3 min and the numbers mean"
      " NOTHING;\nit only proves that every cell works. For the real results set FAST = False and Save & Run All."
      if FAST else f"FULL RUN (FAST = False): planned about 4 h. A training job that ends after a few"
      " minutes FAILED:\nread the '----- ... failed' lines below the cell and logs/<run>.log.")
print("=" * 100)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.8/320.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 5.5 MB/s eta 0:00:00
torch 2.11.0+cu128 | GPUs: ['Tesla T4', 'Tesla T4']
FULL RUN (FAST = False): planned about 4 h. A training job that ends after a few minutes FAILED:
read the '----- ... failed' lines below the cell and logs/<run>.log.


## 1. Write the code (`lprkit/`)
The same files are in this folder of the team repo; the cells below only write them to `/kaggle/working/lprkit`.

In [2]:
%%writefile /kaggle/working/lprkit/common.py
"""Shared helpers for the 498R paper notebooks (Kaggle GPU T4 x2).

  find_dataset()          -> folder that holds dataset_manifest.json (the "498R BanglaLPR All-in-One" dataset)
  data_yaml(part, kind)   -> absolute-path YOLO yaml for detection / recognition (kind 'all' or 'human')
  run_on_gpus(jobs)       -> start one command per GPU at the same time, wait, return exit codes
  env_info(), save_json(), load_json(), timer
"""
import glob
import json
import os
import platform
import subprocess
import sys
import time

WORK = os.environ.get('LPR_WORK', '/kaggle/working')
WORKERS = int(os.environ.get('LPR_WORKERS', 4))     # DataLoader workers (0 for local Windows tests)


# The team copy of the dataset (Google Drive folder "All In One 1.1", shared with the team). v1.0 (runs up to 20 Oct
# 2026) stays in folder "All In One" (1RLwSI5_YINfJr7tymuKBa8Hxkp0dKKSK), file id 1GGVA0Plxxh6uvRuiDVGEOjiWENHXxmv-.
DRIVE_FOLDER = 'https://drive.google.com/drive/folders/11psdzW5Cpr7tXwqzqNxNt3LA-qeDSOQl'
DRIVE_ZIP_ID = '1PPM6ONnkXYkBNJ5DEM94Pa941cBwbDOg'          # BanglaLPR_AllInOne_v1_1.zip (v1.1, 3.63 GB)
DATASET_VERSION = '1.1'                                     # "version" in dataset_manifest.json that the kits expect
DOWNLOAD_DIR = '/tmp/498r_data' if os.path.exists('/kaggle') else os.path.expanduser('~/498r_data')
_version_warned = set()


def _check_version(root):
    """Warn (once per folder) when the dataset found is not the expected version, e.g. an old v1.0 Kaggle input or an
    old unpacked copy, which find_dataset() would otherwise use without a word."""
    try:
        v = json.load(open(os.path.join(root, 'dataset_manifest.json'), encoding='utf-8')).get('version', '?')
    except (OSError, ValueError):
        v = '?'
    if v != DATASET_VERSION and root not in _version_warned:
        _version_warned.add(root)
        print(f'WARNING: the dataset at {root} is version {v}, these notebooks expect v{DATASET_VERSION}. Remove that '
              f'input / folder to download v{DATASET_VERSION} from {DRIVE_FOLDER}, unless you re-run an old result.',
              flush=True)
    return root


def _search():
    for base in ('/kaggle/input', DOWNLOAD_DIR, os.path.expanduser('~/data'), '/content', '.'):
        hits = sorted(glob.glob(os.path.join(base, '**', 'dataset_manifest.json'), recursive=True), key=len)
        if hits:
            return os.path.dirname(hits[0])
    return None


def download_dataset(dest=DOWNLOAD_DIR, file_id=DRIVE_ZIP_ID):
    """Download the dataset zip from the team Google Drive and unpack it (about 3-6 min on Kaggle). It goes to /tmp on
    Kaggle, so the 3.6 GB are NOT saved as notebook output."""
    import shutil
    import urllib.request
    import zipfile
    os.makedirs(dest, exist_ok=True)
    z = os.path.join(dest, 'BanglaLPR_AllInOne.zip')
    url = f'https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t'
    print(f'dataset not attached -> downloading it from the team Drive ({DRIVE_FOLDER})', flush=True)
    for attempt in range(3):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64)'})
            with urllib.request.urlopen(req, timeout=120) as r, open(z, 'wb') as f:
                if 'text/html' in r.headers.get('Content-Type', ''):
                    raise OSError('Google Drive sent a web page instead of the file (download quota reached?)')
                total, done, step = int(r.headers.get('Content-Length') or 0), 0, 0
                while True:
                    b = r.read(1 << 22)
                    if not b:
                        break
                    f.write(b); done += len(b)
                    if done >= step:
                        print(f'  {done / 2 ** 30:.2f} / {total / 2 ** 30:.2f} GB', flush=True); step += 500 * 2 ** 20
            if total and os.path.getsize(z) != total:
                raise OSError('incomplete download')
            break
        except OSError as e:
            print(f'  download try {attempt + 1} failed: {e}', flush=True)
    else:
        raise FileNotFoundError('Could not download the dataset from Google Drive. Attach it instead: Add Input -> '
                                'Datasets -> "498r-bangla-lpr-all-in-one" (or upload the zip from the Drive folder '
                                f'{DRIVE_FOLDER} as a Kaggle dataset).')
    print('  unpacking ...', flush=True)
    with zipfile.ZipFile(z) as f:
        f.extractall(dest)
    os.remove(z)
    root = _search()
    print('dataset ready:', root, flush=True)
    return root


def find_dataset(download=True):
    """The 498R dataset root (folder with dataset_manifest.json): $DATASET_ROOT, an attached Kaggle input, a local copy
    (~/data, /content, ./), else it is downloaded once from the team Google Drive."""
    root = os.environ.get('DATASET_ROOT')
    if root and os.path.exists(os.path.join(root, 'dataset_manifest.json')):
        return _check_version(root)
    root = _search()
    if root:
        return _check_version(root)
    if download and not os.environ.get('LPR_NO_DOWNLOAD'):
        root = download_dataset()
        if root:
            os.environ['DATASET_ROOT'] = root           # jobs started from here inherit it (no second download)
            return _check_version(root)
    raise FileNotFoundError('dataset_manifest.json not found. On Kaggle: Add Input -> "498r-bangla-lpr-all-in-one", '
                            f'or keep Internet on so it downloads from {DRIVE_FOLDER}. Locally: set DATASET_ROOT.')


def data_yaml(part, kind='all', out_dir=None):
    """Copy detection/recognition data(.human).yaml with an absolute `path:` so Ultralytics can read it anywhere."""
    root = find_dataset()
    src = os.path.join(root, part, 'data.yaml' if kind == 'all' else 'data_human.yaml')
    out_dir = out_dir or os.path.join(WORK, 'yaml')
    os.makedirs(out_dir, exist_ok=True)
    lines = open(src, encoding='utf-8').read().splitlines()
    lines = [l for l in lines if not l.startswith('path:') and not l.startswith('#')]
    dst = os.path.join(out_dir, f'{part}_{kind}.yaml')
    with open(dst, 'w', encoding='utf-8') as f:
        f.write(f'path: {os.path.join(root, part)}\n' + '\n'.join(lines) + '\n')
    return dst


def cv2_with_cascade():
    """OpenCV 5 moved the Haar cascades out of the main package; papers #3 and #11 need them. If the installed cv2 has
    no CascadeClassifier, OpenCV 4.x is installed into WORK/opencv4 and imported instead. Call it BEFORE any other
    `import cv2` in the process."""
    probe = subprocess.run([sys.executable, '-c', 'import cv2, sys; sys.exit(0 if hasattr(cv2, "CascadeClassifier") else 1)'])
    if probe.returncode != 0:
        d = os.path.join(WORK, 'opencv4')
        if not os.path.exists(os.path.join(d, 'cv2')):
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--target', d,
                            'opencv-python-headless<5'], check=True)
        sys.path.insert(0, d)
    import cv2
    assert hasattr(cv2, 'CascadeClassifier'), 'this OpenCV has no Haar cascade support'
    return cv2


def gpu_count():
    try:
        import torch
        return torch.cuda.device_count()
    except Exception:
        return 0


def env_info():
    info = {'python': sys.version.split()[0], 'platform': platform.platform(), 'date': time.strftime('%Y-%m-%d %H:%M')}
    try:
        import torch
        info['torch'] = torch.__version__
        info['cuda'] = torch.version.cuda
        info['gpus'] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    except Exception:
        pass
    for mod in ('ultralytics', 'transformers', 'numpy', 'cv2', 'onnxruntime', 'tensorrt', 'timm', 'peft'):
        try:
            m = __import__(mod)
            info[mod] = getattr(m, '__version__', '?')
        except Exception:
            pass
    return info


def save_json(obj, path):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=1, default=str)


def load_json(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)


def run_on_gpus(jobs, log_dir=None):
    """jobs: list of (name, gpu_id or None, [cmd...], extra_env dict or None). All start together; returns {name: code}.
    Each job gets CUDA_VISIBLE_DEVICES=<gpu_id>, so inside the job the GPU is always device 0.
    Jobs given the same gpu_id run one after the other on that GPU (a queue per GPU)."""
    log_dir = log_dir or os.path.join(WORK, 'logs')
    os.makedirs(log_dir, exist_ok=True)
    codes = {}

    def start(job):
        name, gpu, cmd, extra = (list(job) + [None])[:4]
        env = dict(os.environ)
        if gpu is not None:
            env['CUDA_VISIBLE_DEVICES'] = str(gpu)
        env.update(extra or {})
        log = open(os.path.join(log_dir, f'{name}.log'), 'w', encoding='utf-8')
        print(f'[start] {name} on GPU {gpu}: {" ".join(map(str, cmd))}', flush=True)
        return subprocess.Popen([str(c) for c in cmd], stdout=log, stderr=subprocess.STDOUT, env=env), log, time.time()

    pending = {}
    for job in jobs:
        pending.setdefault(job[1], []).append(job)
    active = {}
    for g in pending:
        if pending[g]:
            job = pending[g].pop(0)
            active[g] = (job,) + start(job)
    while active:
        time.sleep(10)
        for g in list(active):
            job, proc, log, t0 = active[g]
            if proc.poll() is not None:
                log.close()
                codes[job[0]] = proc.returncode
                print(f'[done]  {job[0]} code={proc.returncode} after {(time.time() - t0) / 60:.1f} min', flush=True)
                del active[g]
                if pending[g]:
                    nxt = pending[g].pop(0)
                    active[g] = (nxt,) + start(nxt)
    return codes


def tail(path, n=20):
    try:
        lines = open(path, encoding='utf-8', errors='replace').read().replace('\r', '\n').splitlines()
        return '\n'.join(l for l in lines if l.strip())[-4000:] if n is None else '\n'.join([l for l in lines if l.strip()][-n:])
    except FileNotFoundError:
        return ''


class Timer:
    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *a):
        self.minutes = (time.time() - self.t0) / 60

Writing /kaggle/working/lprkit/common.py


In [3]:
%%writefile /kaggle/working/lprkit/vocab.py
"""Canonical Bangla plate vocabulary for the 498R merged recognition set, and the maps from every source's class names.

Every mapping below was checked by eye on crop galleries (logs/peek/cls_kuet.jpg, cls_blpd.jpg, cls_lpdba.jpg) and,
for BLPD, against the English/Bangla ground truth pairs in its external-validation CSV.
All strings are Unicode NFC (NFC keeps য়/ড়/ঢ় decomposed, consistently).
"""
import unicodedata

N = lambda s: unicodedata.normalize('NFC', s)

DIGITS = list('০১২৩৪৫৬৭৮৯')
METRO = 'মেট্রো'
# vehicle-class (series) letters seen on BRTA plates in our sources
LETTERS = ['অ', 'ই', 'উ', 'এ', 'ক', 'খ', 'গ', 'ঘ', 'ঙ', 'চ', 'ছ', 'জ', 'ঝ', 'ট', 'ঠ', 'ড', 'ঢ', 'ত', 'থ', 'দ', 'ধ',
           'ন', 'প', 'ফ', 'ব', 'ভ', 'ম', 'য', 'র', 'ল', 'শ', 'স', 'হ']
# the 64 districts, official Bangla spelling
DISTRICTS = ['ঢাকা', 'গাজীপুর', 'নারায়ণগঞ্জ', 'নরসিংদী', 'মানিকগঞ্জ', 'মুন্সিগঞ্জ', 'টাঙ্গাইল', 'কিশোরগঞ্জ', 'ফরিদপুর',
             'মাদারীপুর', 'গোপালগঞ্জ', 'রাজবাড়ী', 'শরীয়তপুর',
             'চট্টগ্রাম', 'কক্সবাজার', 'রাঙ্গামাটি', 'বান্দরবান', 'খাগড়াছড়ি', 'ফেনী', 'নোয়াখালী', 'লক্ষ্মীপুর', 'কুমিল্লা',
             'চাঁদপুর', 'ব্রাহ্মণবাড়িয়া',
             'রাজশাহী', 'নাটোর', 'নওগাঁ', 'চাঁপাইনবাবগঞ্জ', 'পাবনা', 'সিরাজগঞ্জ', 'বগুড়া', 'জয়পুরহাট',
             'খুলনা', 'বাগেরহাট', 'সাতক্ষীরা', 'যশোর', 'নড়াইল', 'মাগুরা', 'ঝিনাইদহ', 'কুষ্টিয়া', 'চুয়াডাঙ্গা', 'মেহেরপুর',
             'বরিশাল', 'ভোলা', 'পটুয়াখালী', 'বরগুনা', 'ঝালকাঠি', 'পিরোজপুর',
             'সিলেট', 'মৌলভীবাজার', 'হবিগঞ্জ', 'সুনামগঞ্জ',
             'রংপুর', 'দিনাজপুর', 'ঠাকুরগাঁও', 'পঞ্চগড়', 'নীলফামারী', 'লালমনিরহাট', 'কুড়িগ্রাম', 'গাইবান্ধা',
             'ময়মনসিংহ', 'জামালপুর', 'শেরপুর', 'নেত্রকোণা']
# short metro forms printed on plates (Chattogram / Rajshahi metro) and the pre-2014 name of Chapainawabganj
SHORT = ['চট্ট', 'রাজ', 'নবাবগঞ্জ']
CLASSES = [N(x) for x in DIGITS + [METRO] + LETTERS + DISTRICTS + SHORT]
assert len(DISTRICTS) == 64 and len(set(CLASSES)) == len(CLASSES)
CLASS_ID = {c: i for i, c in enumerate(CLASSES)}
KIND = {N(c): ('digit' if c in DIGITS else 'metro' if c == METRO else 'letter' if c in LETTERS else 'district')
        for c in DIGITS + [METRO] + LETTERS + DISTRICTS + SHORT}

# ---- spelling variants -> canonical
VARIANTS = {'নারাইল': 'নড়াইল', 'পটুয়াখালি': 'পটুয়াখালী', 'নীলফামারি': 'নীলফামারী', 'মাদারিপুর': 'মাদারীপুর',
            'শরিয়তপুর': 'শরীয়তপুর', 'চাপাইনবাবগঞ্জ': 'চাঁপাইনবাবগঞ্জ', 'রাঙামাটি': 'রাঙ্গামাটি', 'চট্টো': 'চট্ট',
            'নেত্রকোনা': 'নেত্রকোণা', 'বগুরা': 'বগুড়া', '0': '০'}

# ---- KUET (Hossain & Hassan) English names, checked on cls_kuet.jpg
KUET = {**{str(i): DIGITS[i] for i in range(10)}, 'ka': 'ক', 'kha': 'খ', 'ga': 'গ', 'gha': 'ঘ', 'ca': 'চ', 'ja': 'জ',
        'ta': 'ট', 'da': 'ড', 'na': 'ন', 'ba': 'ব', 'va': 'ভ', 'ha': 'হ', 'metro': METRO, 'dhaka': 'ঢাকা',
        'khulna': 'খুলনা', 'chattogram': 'চট্ট', 'jashore': 'যশোর'}

# ---- BLPD (Sarker & Alam) English names, checked on cls_blpd.jpg and the external-validation CSV
BLPD = {**{str(i): DIGITS[i] for i in range(10)},
        'A': 'অ', 'Ba': 'ব', 'Bha': 'ভ', 'Cha': 'চ', 'Chha': 'ছ', 'DA': 'দ', 'Da': 'ড', 'Dha': 'ঢ', 'E': 'এ',
        'Ga': 'গ', 'Gha': 'ঘ', 'Ha': 'হ', 'Ja': 'জ', 'Jha': 'ঝ', 'Ka': 'ক', 'Kha': 'খ', 'La': 'ল', 'Ma': 'ম',
        'Na': 'ন', 'Pa': 'প', 'Sa': 'স', 'Sha': 'শ', 'THA': 'ঠ', 'Ta': 'ট', 'Tha': 'থ', 'U': 'উ',
        'Metro': METRO, 'Chatto': 'চট্ট', 'Raj': 'রাজ', 'Nowabganj': 'নবাবগঞ্জ',
        'Bagerhat': 'বাগেরহাট', 'Bagura': 'বগুড়া', 'Bandarban': 'বান্দরবান', 'Barguna': 'বরগুনা', 'Barisal': 'বরিশাল',
        'Bhola': 'ভোলা', 'Brahmanbaria': 'ব্রাহ্মণবাড়িয়া', 'Chandpur': 'চাঁদপুর', 'Chapainawabganj': 'চাঁপাইনবাবগঞ্জ',
        'Chattogram': 'চট্টগ্রাম', 'Chuadanga': 'চুয়াডাঙ্গা', 'Cox-s Bazar': 'কক্সবাজার', 'Cox-s-Bazar': 'কক্সবাজার',
        'Cumilla': 'কুমিল্লা', 'Dhaka': 'ঢাকা', 'Dinajpur': 'দিনাজপুর', 'Faridpur': 'ফরিদপুর', 'Feni': 'ফেনী',
        'Gaibandha': 'গাইবান্ধা', 'Gazipur': 'গাজীপুর', 'Gopalganj': 'গোপালগঞ্জ', 'Habiganj': 'হবিগঞ্জ',
        'Jamalpur': 'জামালপুর', 'Jessore': 'যশোর', 'Jhalokati': 'ঝালকাঠি', 'Jhenaidah': 'ঝিনাইদহ',
        'Joypurhat': 'জয়পুরহাট', 'Khagrachari': 'খাগড়াছড়ি', 'Khulna': 'খুলনা', 'Kishoreganj': 'কিশোরগঞ্জ',
        'Kurigram': 'কুড়িগ্রাম', 'Kustia': 'কুষ্টিয়া', 'Lakshmipur': 'লক্ষ্মীপুর', 'Lalmonirhat': 'লালমনিরহাট',
        'Madaripur': 'মাদারীপুর', 'Magura': 'মাগুরা', 'Manikganj': 'মানিকগঞ্জ', 'Meherpur': 'মেহেরপুর',
        'Moulvibazar': 'মৌলভীবাজার', 'Munshiganj': 'মুন্সিগঞ্জ', 'Mymensingh': 'ময়মনসিংহ', 'Naogaon': 'নওগাঁ',
        'Narail': 'নড়াইল', 'Narayanganj': 'নারায়ণগঞ্জ', 'Narsingdi': 'নরসিংদী', 'Natore': 'নাটোর',
        'Netrokona': 'নেত্রকোণা', 'Nilphamari': 'নীলফামারী', 'Noakhali': 'নোয়াখালী', 'Pabna': 'পাবনা',
        'Patuakhali': 'পটুয়াখালী', 'Pirojpur': 'পিরোজপুর', 'Rajbari': 'রাজবাড়ী', 'Rajshahi': 'রাজশাহী',
        'Rangamati': 'রাঙ্গামাটি', 'Rangpur': 'রংপুর', 'Satkhira': 'সাতক্ষীরা', 'Shariatpur': 'শরীয়তপুর',
        'Sherpur': 'শেরপুর', 'Sirajganj': 'সিরাজগঞ্জ', 'Sunamganj': 'সুনামগঞ্জ', 'Sylhet': 'সিলেট',
        'Tangail': 'টাঙ্গাইল', 'Thakurgaon': 'ঠাকুরগাঁও', 'panchagarh': 'পঞ্চগড়'}

# ---- NightLPBD transliterations, checked on cls_night.jpg (syllable boxes are dropped in ingest.py)
NIGHT = {**{str(i): DIGITS[i] for i in range(10)}, 'bo': 'ব', 'co': 'চ', 'go': 'গ', 'ho': 'হ', 'To': 'ট', 'lo': 'ল',
         'Dhaka': 'ঢাকা', 'khulona': 'খুলনা', 'meTrO': METRO, 'zoshOro': 'যশোর'}


# ---- furcifer/bangla-license-plates-synthetic (Kaggle, CC0) lp.names, checked on rendered samples (thaa = থ, daa = দ)
FURCIFER = {**{str(i): DIGITS[i] for i in range(10)}, 'a': 'অ', 'i': 'ই', 'u': 'উ', 'e': 'এ', 'ka': 'ক', 'kha': 'খ',
            'ga': 'গ', 'gha': 'ঘ', 'umno': 'ঙ', 'ca': 'চ', 'cha': 'ছ', 'ja': 'জ', 'jha': 'ঝ', 'tSa': 'ট', 'tSha': 'ঠ',
            'dSa': 'ড', 'dSha': 'ঢ', 'thaa': 'থ', 'daa': 'দ', 'na': 'ন', 'pa': 'প', 'pha': 'ফ', 'ba': 'ব', 'bha': 'ভ',
            'ma': 'ম', 'jSaa': 'য', 'ra': 'র', 'la': 'ল', 'sha': 'শ', 'sa': 'স', 'ha': 'হ', 'dhaka': 'ঢাকা',
            'chittagong': 'চট্টগ্রাম', 'khulna': 'খুলনা', 'kushtia': 'কুষ্টিয়া', 'narayanganj': 'নারায়ণগঞ্জ',
            'bagura': 'বগুড়া', 'comilla': 'কুমিল্লা', 'mymensingh': 'ময়মনসিংহ', 'sylhet': 'সিলেট'}

def canon(source, name):
    """Map a source class name to a canonical class (or None if it cannot be mapped)."""
    if source == 'kuet':
        v = KUET.get(name)
    elif source == 'blpd':
        v = BLPD.get(name)
    elif source == 'nightlpbd':
        v = NIGHT.get(name)
    elif source == 'furcifer':
        v = FURCIFER.get(name)
    else:                                   # lpdba, foggy_bd: Bangla names already
        v = name
    if v is None:
        return None
    v = N(VARIANTS.get(N(v), v))
    v = N(VARIANTS.get(v, v))
    return v if v in CLASS_ID else None


def plate_text(tokens_with_boxes):
    """Plate string from canonical tokens + boxes [[token, x1, y1, x2, y2], ...].
    BRTA layout: top row = district (or short metro form) [মেট্রো] - letter, bottom row = 6 digits (NN-NNNN).
    Rows are taken from the token kind (words vs digits), not from y, so tilted plates still work; y is only used
    to check that the words sit above the digits. Returns (text, valid) where valid = exactly that layout."""
    words = [t for t in tokens_with_boxes if KIND.get(t[0]) == 'district']
    metro = [t for t in tokens_with_boxes if KIND.get(t[0]) == 'metro']
    letters = [t for t in tokens_with_boxes if KIND.get(t[0]) == 'letter']
    digits = [t for t in tokens_with_boxes if KIND.get(t[0]) == 'digit']
    cx = lambda t: (t[1] + t[3]) / 2
    cy = lambda t: (t[2] + t[4]) / 2
    top = ' '.join(t[0] for t in sorted(words, key=cx))
    if metro:
        top = (top + ' ' if top else '') + METRO
    if letters:
        top = (top + '-' if top else '') + ''.join(t[0] for t in sorted(letters, key=cx))
    ds = ''.join(t[0] for t in sorted(digits, key=cx))
    bottom = ds[:2] + '-' + ds[2:] if len(ds) == 6 else ds
    text = (top + ' ' + bottom).strip()
    up = words + metro + letters
    valid = (len(words) == 1 and len(metro) <= 1 and len(letters) == 1 and len(digits) == 6
             and sum(map(cy, up)) / len(up) < sum(map(cy, digits)) / len(digits))
    return text, valid


def norm_text(s):
    """Text used for scoring: NFC, Bangla digits, no spaces/hyphens/dots."""
    s = N(s)
    s = s.translate(str.maketrans('0123456789', '০১২৩৪৫৬৭৮৯'))
    for ch in ' -–—.:,_':
        s = s.replace(ch, '')
    return s


# English transliteration for each canonical class (from the BLPD names, which match BRTA English usage)
ENGLISH = {N(v): k for k, v in BLPD.items() if not k.isdigit() and k not in ('Cox-s-Bazar', 'DA')}
ENGLISH.update({N('দ'): 'DA', N('কক্সবাজার'): "Cox's Bazar", N('ই'): 'I', N('ঙ'): 'Umo', N('ত'): 'Ta (dental)',
                N('ধ'): 'Dha (dental)', N('ফ'): 'Pha', N('য'): 'Ya', N('র'): 'Ra', N('নবাবগঞ্জ'): 'Nawabganj'})
ENGLISH.update({N(d): str(i) for i, d in enumerate(DIGITS)})
ENGLISH[N(METRO)] = 'Metro'


def text_tokens(text):
    """Split a canonical plate text ("ঢাকা মেট্রো-গ ১২-৩৪৫৬") into class tokens: district, [মেট্রো], letter, 6 digits."""
    t = N(text)
    out = []
    head, _, digits = t.rpartition(' ')
    if '-' in head:
        words, _, letter = head.rpartition('-')
    else:
        words, letter = head, ''
    for w in words.split():
        out.append(w)
    if letter:
        out.append(letter)
    out.extend(ch for ch in digits if ch in DIGITS)
    return out


def tokens_to_text(toks):
    """Inverse of text_tokens for a predicted token sequence (any order): district(s) [মেট্রো]-letter DD-DDDD."""
    words = [t for t in toks if KIND.get(t) == 'district']
    metro = [t for t in toks if KIND.get(t) == 'metro']
    letters = [t for t in toks if KIND.get(t) == 'letter']
    digits = ''.join(t for t in toks if KIND.get(t) == 'digit')
    top = ' '.join(words)
    if metro:
        top = (top + ' ' if top else '') + METRO
    if letters:
        top = (top + '-' if top else '') + ''.join(letters)
    bottom = digits[:2] + '-' + digits[2:] if len(digits) == 6 else digits
    return (top + ' ' + bottom).strip()


PUA = 0xE000   # one private-use symbol per token, for libraries that need a 1-character alphabet (PARSeq, fast-plate-ocr)


def sym_alphabet(classes):
    return {c: chr(PUA + i) for i, c in enumerate(classes)}


def text_to_syms(text, classes):
    m = sym_alphabet(classes)
    return ''.join(m[t] for t in text_tokens(text) if t in m)


def syms_to_text(syms, classes):
    inv = {v: k for k, v in sym_alphabet(classes).items()}
    return tokens_to_text([inv[c] for c in syms if c in inv])

Writing /kaggle/working/lprkit/vocab.py


In [4]:
%%writefile /kaggle/working/lprkit/metrics.py
"""OCR scoring and fusion for Bangla plates.

Texts are compared after vocab.norm_text (NFC, Bangla digits, no spaces/hyphens), so "ঢাকা মেট্রো-গ ১২-৩৪৫৬" and
"ঢাকামেট্রোগ১২৩৪৫৬" are the same plate. Three views of one prediction:
  plate exact  : whole plate right
  CER          : Unicode character edit distance / truth length (what papers #1, #32 report)
  token acc    : 1 - token edit distance / truth tokens; tokens = district, মেট্রো, letter, 6 digits (vocab.text_tokens)
  WER          : word edit distance / truth words (words = parts between spaces and hyphens)
"""
import collections

try:
    from . import vocab
except ImportError:
    import vocab


def lev(a, b):
    """Edit distance between two sequences (str or list)."""
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]


def tokens(text):
    t = vocab.N(text or '')
    try:
        return vocab.text_tokens(t)
    except Exception:
        return list(vocab.norm_text(t))


def words(text):
    """Words for WER (papers #1, #32): the plate text split at spaces and hyphens, digits as one word per group."""
    t = vocab.N(text or '')
    for d, b in zip('0123456789', vocab.DIGITS):
        t = t.replace(d, b)
    return t.replace('-', ' ').split()


def score_rows(rows):
    """rows: iterable of (pred_text, true_text). Returns aggregate metrics in %."""
    n = exact = ce = cl = te = tl = we = wl = 0
    for pred, true in rows:
        p, t = vocab.norm_text(pred or ''), vocab.norm_text(true)
        pw, tw = words(pred), words(true)
        we += lev(pw, tw)
        wl += max(1, len(tw))
        n += 1
        exact += int(p == t)
        ce += lev(p, t)
        cl += max(1, len(t))
        pt, tt = tokens(pred), tokens(true)
        te += lev(pt, tt)
        tl += max(1, len(tt))
    return {'plates': n, 'plate_exact_%': round(100 * exact / max(1, n), 2), 'CER_%': round(100 * ce / max(1, cl), 2),
            'char_acc_%': round(max(0.0, 100 * (1 - ce / max(1, cl))), 2), 'token_acc_%': round(max(0.0, 100 * (1 - te / max(1, tl))), 2),
            'WER_%': round(100 * we / max(1, wl), 2)}


def fuse(preds, how='MV-HC', best_model=None):
    """preds: list of (model_name, text, confidence) for ONE plate. Returns the fused text.
    HC: most confident; MV-HC / MV-BM: most frequent normalised string, ties by confidence / by best_model;
    MVCP-HC: vote per token position (tokens padded to the longest), ties by confidence.
    MV-BM+format (ours, not in paper #24): drop readings that break the BRTA layout (is_valid_plate), then MV-BM;
    if no reading fits the layout, MV-BM on all of them."""
    if not preds:
        return ''
    if how == 'HC':
        return max(preds, key=lambda x: x[2])[1]
    if how == 'MV-BM+format':
        return fuse([p for p in preds if is_valid_plate(p[1])] or preds, 'MV-BM', best_model)
    if how.startswith('MV') and not how.startswith('MVCP'):
        groups = collections.defaultdict(list)
        for m, t, c in preds:
            groups[vocab.norm_text(t)].append((m, t, c))
        top = max(len(v) for v in groups.values())
        tied = [v for v in groups.values() if len(v) == top]
        if len(tied) == 1:
            return tied[0][0][1]
        if how == 'MV-BM' and best_model is not None:
            for v in tied:
                if any(m == best_model for m, _, _ in v):
                    return v[0][1]
        return max(tied, key=lambda v: sum(c for _, _, c in v) / len(v))[0][1]
    if how.startswith('MVCP'):
        toks = [(tokens(t), c) for _, t, c in preds]
        L = max(len(t) for t, _ in toks)
        out = []
        for i in range(L):
            votes = collections.defaultdict(list)
            for t, c in toks:
                votes[t[i] if i < len(t) else ''].append(c)
            top = max(len(v) for v in votes.values())
            cands = [(k, max(v)) for k, v in votes.items() if len(v) == top]
            out.append(max(cands, key=lambda kv: kv[1])[0])
        toks_out = [x for x in out if x]
        return ' '.join(toks_out)
    raise ValueError(how)


EXTRA_TESTS = ('test_fog', 'test_rain_synth')     # extra OCR test splits of the 498R dataset (ocr/<name>.csv)


def condition_scores(preds, root):
    """preds: dicts with file / pred / true on the real OCR test split -> scores per condition tag
    (ocr/test_conditions.csv of the 498R dataset: clear, night, blurry, low_res). None if the file is missing."""
    import csv
    import os
    f = os.path.join(root, 'ocr', 'test_conditions.csv')
    if not os.path.exists(f):
        return None
    tags = {r['file']: r for r in csv.DictReader(open(f, encoding='utf-8'))}
    out = {}
    for tag in ('clear', 'night', 'blurry', 'low_res'):
        rows = [(p['pred'], p['true']) for p in preds if tags.get(p['file'], {}).get(tag) == '1']
        if rows:
            out[tag] = score_rows(rows)
    return out


def extra_splits(root):
    import os
    return [s for s in EXTRA_TESTS if os.path.exists(os.path.join(root, 'ocr', f'{s}.csv'))]


def is_valid_plate(text):
    """True if the text parses as one BRTA plate: district [মেট্রো] letter + 6 digits."""
    tk = tokens(text)
    kinds = [vocab.KIND.get(t, '?') for t in tk]
    return (kinds.count('district') == 1 and kinds.count('letter') == 1 and kinds.count('digit') == 6
            and kinds.count('metro') <= 1 and '?' not in kinds)


def mcnemar_exact(b, c):
    """Exact two-sided McNemar test on paired plates: b = plates only system A reads right, c = only system B.
    Binomial test of min(b, c) against Bin(b + c, 0.5). Returns p (1.0 when the two never disagree)."""
    import math
    n = b + c
    if n == 0:
        return 1.0
    p = sum(math.comb(n, i) for i in range(min(b, c) + 1)) / 2 ** n
    return round(min(1.0, 2 * p), 4)

Writing /kaggle/working/lprkit/metrics.py


In [5]:
%%writefile /kaggle/working/lprkit/report.py
"""Write results_vs_paper.md and pack the outputs zip at the end of a paper notebook.

paper_rows: list of (metric label, paper value as text, page, key) where key is a dotted path inside a run's
results.json (e.g. 'test.mAP50', 'ocr_test.rows_decoder.CER_%', 'speed.fp32.fps'); values in 0..1 for
P/R/mAP are shown in %. 'a|b' = the first of a, b with a number (e.g. 'speed.tensorrt_fp16.fps|speed.fp32.fps':
TensorRT, else PyTorch FP32 when the export failed); a fallback value is marked with its source, e.g. '52.90 (fp32)'.
"""
import glob
import json
import os
import shutil
import zipfile

try:
    from . import common
except ImportError:
    import common

PCT_KEYS = ('P', 'R', 'mAP50', 'mAP50-95')


def get(d, key):
    for k in key.split('.'):
        if not isinstance(d, dict) or k not in d:
            return None
        d = d[k]
    return d


def fmt(v, key):
    if v is None:
        return '–'
    if isinstance(v, (int, float)):
        last = key.split('.')[-1]
        if last in PCT_KEYS and v <= 1.0:
            return f'{100 * v:.1f}'
        if isinstance(v, float):
            return f'{v:.4f}' if 0 < abs(v) < 0.1 else f'{v:.2f}'
        return str(v)
    return str(v)[:60]


def cell(d, key):
    """(text, used a fallback) for key 'a|b|...': the first alternative with a number; if none has one, the first value
    found (e.g. 'failed: ...' when the TensorRT export failed)."""
    alts = key.split('|')
    for i, k in enumerate(alts):
        v = get(d, k)
        if isinstance(v, (int, float)) and not isinstance(v, bool):
            return (fmt(v, k), False) if i == 0 else (f"{fmt(v, k)} ({k.split('.')[-2] if '.' in k else k})", True)
    v = next((x for x in (get(d, k) for k in alts) if x is not None), None)
    return fmt(v, alts[0]), False


def compare_table(runs_dir, paper_rows, runs, title, notes='', same_data=False):
    """key 'run::path' fills only that run's column (for rows that belong to one run)."""
    res = {}
    for r in runs:
        p = os.path.join(runs_dir, r, 'results.json')
        if os.path.exists(p):
            res[r] = common.load_json(p)
    cols = ['Metric', 'Paper (page)'] + list(res)
    lines = [f'# {title}', '', '| ' + ' | '.join(cols) + ' |', '|' + '---|' * len(cols)]
    fallbacks = []
    for label, pval, page, key in paper_rows:
        row = [label, f'{pval} ({page})' if page else str(pval)]
        only, k = key.split('::', 1) if key and '::' in key else (None, key)
        for r, d in res.items():
            if k and (only is None or only == r):
                text, fb = cell(d, k)
                row.append(text)
                if fb and k not in fallbacks:
                    fallbacks.append(k)
            else:
                row.append('')
        lines.append('| ' + ' | '.join(row) + ' |')
    for k in fallbacks:
        first, rest = k.split('|', 1)
        lines += ['', f'A value marked with (...) comes from `{rest}`: that run has no number at `{first}` '
                      '(e.g. the TensorRT export failed or was not run).']
    missing = [r for r in runs if r not in res]
    if missing:
        lines += ['', f'Runs without results.json (failed or not run): {", ".join(missing)}']
    if same_data:
        lines += ['', "Same data and split as the paper (its own published data / weights), so this IS a direct check of",
                  "the paper's numbers. Remaining differences come from software versions, hardware and randomness."]
    else:
        lines += ['', 'The paper used its own data; our numbers are on the 498R All-in-One test split (different data),',
                  'so the comparison shows the direction and size of the gap, not a head-to-head result.']
    if notes:
        lines += ['', notes]
    return '\n'.join(lines) + '\n'


def condition_table(runs_dir, runs):
    """Results by condition (498R condition test sets): detectors -> mAP50 % per subset and the false-alarm rate on
    plate-free photos; OCR models -> plate exact % per tag, on the real fog crops and on synthetic rain copies."""
    det_cols = ['clear', 'night', 'blurry', 'small_plates', 'rain_synth', 'fog_synth']
    det_rows, ocr_rows = [], []
    for r in runs:
        p = os.path.join(runs_dir, r, 'results.json')
        if not os.path.exists(p):
            continue
        d = common.load_json(p)
        c = d.get('test_by_condition')
        if isinstance(c, dict):
            cells = [fmt(get(c, f'{k}.mAP50'), 'mAP50') for k in det_cols]
            det_rows.append(f"| {r} | {fmt(get(d, 'test.mAP50'), 'mAP50')} | " + ' | '.join(cells) +
                            f" | {fmt(get(c, 'no_plate.false_alarm_%'), 'x')} |")
        o = get(d, 'ocr_test')
        if isinstance(o, dict) and (o.get('by_condition') or o.get('extra_tests')):
            b, e = o.get('by_condition') or {}, o.get('extra_tests') or {}
            base = get(o, 'rows_decoder.plate_exact_%')
            ocr_rows.append(f"| {r} | {fmt(base, 'x')} | " + ' | '.join(fmt(get(b, f'{k}.plate_exact_%'), 'x')
                                                                       for k in ('clear', 'night', 'blurry')) +
                            f" | {fmt(get(e, 'test_fog.plate_exact_%'), 'x')} | {fmt(get(e, 'test_rain_synth.plate_exact_%'), 'x')} |")
    out = []
    if det_rows:
        out += ['', '## Results by condition: detectors (test mAP50 %)', '',
                '| Run | all test | clear | night | blurry | small plates | rain (synthetic) | fog (synthetic) | '
                'false alarms on plate-free photos (%) |', '|' + '---|' * 9] + det_rows
    if ocr_rows:
        out += ['', '## Results by condition: plate readers (plate exact %)', '',
                '| Run | all test | clear | night | blurry | fog (real fog copies) | rain (synthetic) |',
                '|' + '---|' * 7] + ocr_rows
    if out:
        out += ['', 'Condition subsets are fixed for the whole team (detection/conditions.json, ocr/conditions.json); '
                    'rain and detection-fog copies are synthetic.']
    return '\n'.join(out) + ('\n' if out else '')


def pack(work, name, extra_globs=()):
    """Zip runs/*/(weights, results.json, args.yaml, results.csv, preds), logs, the md tables and env info."""
    zpath = os.path.join(work, f'{name}_outputs.zip')
    keep = ('results.json', 'args.yaml', 'results.csv', 'ocr_preds_test.csv', 'best.pt', 'last.pt', 'config.json',
            'model.safetensors', 'results.png', 'confusion_matrix.png', 'labels.jpg', 'BoxPR_curve.png')
    with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in glob.glob(os.path.join(work, 'runs', '**', '*'), recursive=True):
            ok_ext = ('.md', '.json', '.pt', '.ckpt', '.onnx', '.safetensors', '.csv', '.yaml')
            if os.path.isfile(f) and (os.path.basename(f) in keep or f.endswith(ok_ext)) and os.path.getsize(f) < 1.5e9:
                if '/eval_' in f.replace('\\', '/') and not f.endswith('.json'):
                    continue
                z.write(f, os.path.relpath(f, work))
        for pat in ('logs/*.log', '*.md', 'environment.json', 'code/*.py') + tuple(extra_globs):
            for f in glob.glob(os.path.join(work, pat)):
                z.write(f, os.path.relpath(f, work))
    print('packed', zpath, round(os.path.getsize(zpath) / 1e6, 1), 'MB')
    # small copy without model files, for the team GitHub repo (weights/ folder; web upload limit 25 MB per file)
    spath = os.path.join(work, f'{name}_results.zip')
    model_ext = ('.pt', '.ckpt', '.onnx', '.safetensors', '.engine', '.bin', '.pth')
    with zipfile.ZipFile(zpath) as src, zipfile.ZipFile(spath, 'w', zipfile.ZIP_DEFLATED) as dst:
        for n in src.namelist():
            if not n.endswith(model_ext):
                dst.writestr(n, src.read(n))
    print('packed', spath, round(os.path.getsize(spath) / 1e6, 1), 'MB (no weights: for GitHub weights/)')
    return zpath

Writing /kaggle/working/lprkit/report.py


In [6]:
%%writefile /kaggle/working/lprkit/det.py
"""Train + test one plate DETECTOR (or character detector) with Ultralytics, then write results.json.

  python det.py --data det.yaml --model yolov8s.pt --name v8s --project runs
  python det.py --data det.yaml --model rtdetr-l.pt --rtdetr --optimizer AdamW --lr0 1e-4 --batch 8 --name rtdetr
  python det.py --data det.yaml --name v8s --project runs --fps-only [--trt] [--cpu]
  python det.py ... --stages '[{"epochs":12,"freeze":12,"hyp":{...}}, ...]'      (multi-stage schedule, paper #1)

The test split is used once, at the end, with best.pt (best on val). Speed is measured separately (--fps-only) on an
idle GPU at batch 1 so two trainings running at the same time do not distort it.
results.json: settings, epochs, best epoch, minutes, params, GFLOPs, val/test P R mAP50 mAP50-95, plate counts at
conf 0.25 (found %, correct %, recall by plate size, "IoU>0.7 accuracy" of paper #1), speed, versions.
"""
import argparse
import json
import os
import platform
import sys
import time
from pathlib import Path

import numpy as np
import torch
import yaml

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import common


def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)


def split_files(data_yaml, split):
    d = yaml.safe_load(Path(data_yaml).read_text(encoding='utf-8'))
    src = Path(d['path']) / d[split]
    if src.suffix == '.txt':
        imgs = [Path(d['path']) / l.strip() for l in src.read_text(encoding='utf-8').splitlines() if l.strip()]
        img_dir = imgs[0].parent if imgs else src.parent
    else:
        img_dir = src
        imgs = sorted(p for p in img_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
    lab = lambda p: Path(str(p.parent).replace('images', 'labels', 1)) / (p.stem + '.txt')
    return imgs, lab


def gt_boxes(label_file, w, h, cls=None):
    if not label_file.exists():
        return np.zeros((0, 4)), np.zeros(0, int)
    rows = [l.split() for l in label_file.read_text().splitlines() if l.strip()]
    if not rows:
        return np.zeros((0, 4)), np.zeros(0, int)
    c = np.array([int(r[0]) for r in rows])
    b = np.array([[float(v) for v in r[1:5]] for r in rows])
    xyxy = np.stack([(b[:, 0] - b[:, 2] / 2) * w, (b[:, 1] - b[:, 3] / 2) * h,
                     (b[:, 0] + b[:, 2] / 2) * w, (b[:, 1] + b[:, 3] / 2) * h], 1)
    return xyxy, c


def plate_counts(model, data_yaml, split, imgsz, device, conf=0.25, iou_thr=0.5, max_images=None):
    """Plain-language numbers at one working threshold (conf 0.25, IoU 0.5), plus:
       recall by plate size (COCO areas on the stored image: small < 32^2, medium < 96^2, large) and
       paper #1's metric: share of photos where EVERY plate has a box with IoU > 0.7 (iou70_acc_%), mean IoU."""
    imgs, lab = split_files(data_yaml, split)
    imgs = imgs[:max_images] if max_images else imgs
    tp = fp = fn = ok = 0
    size_tp, size_n = {'small': 0, 'medium': 0, 'large': 0}, {'small': 0, 'medium': 0, 'large': 0}
    iou70 = n_with_gt = 0
    ious = []
    src = {}
    for s in range(0, len(imgs), 32):                 # a list source is one batch in Ultralytics 8.4: feed in chunks
        for r in model.predict([str(p) for p in imgs[s:s + 32]], imgsz=imgsz, conf=conf, device=device, verbose=False):
            h, w = r.orig_shape
            g, _ = gt_boxes(lab(Path(r.path)), w, h)
            p = r.boxes.xyxy.cpu().numpy()
            order = np.argsort(-r.boxes.conf.cpu().numpy())
            m = iou_matrix(p[order], g)
            used, t = {}, 0
            for i in range(len(order)):
                j = int(np.argmax(m[i])) if m.shape[1] else -1
                if j >= 0 and m[i, j] >= iou_thr and j not in used:
                    used[j] = m[i, j]
                    t += 1
            tp, fp, fn = tp + t, fp + len(p) - t, fn + len(g) - t
            sname = Path(r.path).name.split('_')[0]
            sc = src.setdefault(sname, [0, 0, 0])
            sc[0] += t; sc[1] += len(g); sc[2] += 1
            ok += int(t == len(g) and t == len(p))
            for j, gb in enumerate(g):
                a = (gb[2] - gb[0]) * (gb[3] - gb[1])
                k = 'small' if a < 32 ** 2 else 'medium' if a < 96 ** 2 else 'large'
                size_n[k] += 1
                size_tp[k] += int(j in used)
            if len(g):
                n_with_gt += 1
                best = m.max(0) if m.shape[0] else np.zeros(len(g))
                ious.extend(best.tolist())
                iou70 += int(bool(np.all(best > 0.7)))
    return {'images': len(imgs), 'plates': tp + fn, 'found_%': round(100 * tp / max(1, tp + fn), 2),
            'correct_%': round(100 * tp / max(1, tp + fp), 2), 'img_all_ok_%': round(100 * ok / max(1, len(imgs)), 2),
            'recall_by_size_%': {k: (round(100 * size_tp[k] / size_n[k], 2) if size_n[k] else None) for k in size_n},
            'plates_by_size': size_n, 'iou70_acc_%': round(100 * iou70 / max(1, n_with_gt), 2),
            'mean_best_iou_%': round(100 * float(np.mean(ious)), 2) if ious else None, 'conf': conf, 'iou': iou_thr,
            'found_%_by_source': {k: {'found_%': round(100 * v[0] / max(1, v[1]), 2), 'plates': v[1], 'images': v[2]}
                                  for k, v in sorted(src.items())}}


def condition_metrics(model, data_yaml, imgsz, device, run_dir, max_images=None):
    """Test results per condition when the dataset has detection/conditions.json (498R All-in-One):
    real subsets (clear, night, blurry, small_plates) and synthetic rain / fog copies of the whole test split -> mAP;
    no_plate photos -> false-alarm rate (share of plate-free photos with a box at conf 0.25)."""
    y = yaml.safe_load(open(data_yaml, encoding='utf-8'))
    root = Path(y['path'])
    cfile = root / 'conditions.json'
    if not cfile.exists():
        return None
    cond = json.loads(cfile.read_text(encoding='utf-8'))
    out = {}
    tmp = Path(run_dir) / 'cond_yaml'
    tmp.mkdir(parents=True, exist_ok=True)
    jobs = [(k, v['list']) for k, v in cond['real'].items() if k != 'no_plate'] + \
           [(k, 'images/' + v['split']) for k, v in cond['synthetic'].items()]
    for name, test in jobs:
        f = tmp / f'{name}.yaml'
        f.write_text(yaml.safe_dump({**{k: y[k] for k in ('path', 'train', 'val', 'names') if k in y}, 'test': test}),
                     encoding='utf-8')
        try:
            out[name] = val_metrics(model, str(f), 'test', imgsz, device, Path(run_dir) / 'cond', batch=16)
            out[name]['plates'] = plate_counts(model, str(f), 'test', imgsz, device, max_images=max_images)['found_%']
        except Exception as e:                       # e.g. an empty subset
            out[name] = f'skipped: {e}'[:120]
    neg = [l.strip() for l in open(root / cond['real']['no_plate']['list'], encoding='utf-8') if l.strip()]
    neg = [str(root / l[2:]) for l in neg][:max_images]
    if neg:
        hits = 0
        for s in range(0, len(neg), 32):
            hits += sum(len(r.boxes) > 0 for r in model.predict(neg[s:s + 32], imgsz=imgsz, conf=0.25, device=device,
                                                                 verbose=False))
        out['no_plate'] = {'photos': len(neg), 'false_alarm_%': round(100 * hits / len(neg), 2)}
    return out


def val_metrics(model, data_yaml, split, imgsz, device, run_dir, batch=16):
    m = model.val(data=data_yaml, split=split, imgsz=imgsz, batch=batch, device=device, plots=True, workers=common.WORKERS,
                  project=str(run_dir), name=f'eval_{split}', exist_ok=True, verbose=False)
    b = m.box
    out = {'P': round(float(b.mp), 4), 'R': round(float(b.mr), 4), 'mAP50': round(float(b.map50), 4),
           'mAP50-95': round(float(b.map), 4)}
    try:
        out['speed_ms'] = {k: round(v, 2) for k, v in m.speed.items()}
    except Exception:
        pass
    return out


def prec(bits):
    """Ultralytics 8.4 uses quantize=16/32; older versions (e.g. the official YOLOv12 fork) use half=True."""
    try:
        from ultralytics.cfg import DEFAULT_CFG_DICT
        if 'quantize' in DEFAULT_CFG_DICT:
            return {'quantize': bits}
    except Exception:
        pass
    return {'half': bits == 16}


def load_model(weights, rtdetr=False):
    if rtdetr:
        from ultralytics import RTDETR
        return RTDETR(str(weights))
    from ultralytics import YOLO
    return YOLO(str(weights))


def legacy_onnx_export():
    """torch >= 2.9 exports with the new dynamo exporter unless told otherwise (it needs onnxscript); older
    Ultralytics code such as the official YOLOv12 fork does not say, so default it to the TorchScript exporter."""
    import inspect
    import torch.onnx as tox
    if getattr(tox.export, '_lpr_legacy', False) or 'dynamo' not in inspect.signature(tox.export).parameters:
        return
    orig = tox.export

    def export(*args, **kw):
        kw.setdefault('dynamo', False)
        return orig(*args, **kw)
    export._lpr_legacy = True
    tox.export = export


def measure_speed(weights, data_yaml, imgsz, device, rtdetr=False, n=200, trt=False, cpu=False):
    """Batch-1 latency on the test images: preprocess + inference + postprocess (NMS included for YOLOs).
    Per precision: ms_per_image / fps (mean), ms_median / fps_median, slow_calls (timed calls > 3x the median, should
    be 0) and ms_breakdown (median of each part). PyTorch letterboxes every photo to its own shape and the first call
    at a new shape is slow (FP16: ~0.6 s per new shape; it made the old FP16 means up to 8x too slow), so every timed
    photo is run once untimed first, per precision. TensorRT and ONNX use one fixed shape."""
    imgs = [str(p) for p in split_files(data_yaml, 'test')[0][:n]]
    out = {'batch': 1, 'imgsz': imgsz, 'images': len(imgs)}

    def bench(model, dev, kw, k):
        timed = imgs if dev != 'cpu' else imgs[:50]
        for f in timed if dev != 'cpu' else timed[:10]:          # warm-up: every letterbox shape seen once
            model.predict(f, imgsz=imgsz, device=dev, verbose=False, **kw)
        ms, parts = [], []
        for f in timed:
            r = model.predict(f, imgsz=imgsz, device=dev, verbose=False, **kw)[0]
            ms.append(sum(r.speed.values()))
            parts.append(r.speed)
        mean, med = float(np.mean(ms)), float(np.median(ms))
        out[k] = {'ms_per_image': round(mean, 2), 'fps': round(1000 / mean, 1),
                  'ms_median': round(med, 2), 'fps_median': round(1000 / med, 1),
                  'slow_calls': int(sum(m > 3 * med for m in ms)),
                  'ms_breakdown': {kk: round(float(np.median([p[kk] for p in parts])), 2) for kk in parts[0]}}

    if torch.cuda.is_available() and not cpu:
        out['device'] = torch.cuda.get_device_name(0)
        for bits in (32, 16):                           # Ultralytics 8.4: quantize=16/32 replaces half=True
            bench(load_model(weights, rtdetr), device, prec(bits), f'fp{bits}')
        if trt:
            try:
                eng = load_model(weights, rtdetr).export(format='engine', imgsz=imgsz, device=device, **prec(16))
                bench(load_model(eng, rtdetr), device, {}, 'tensorrt_fp16')
            except Exception as e:
                out['tensorrt_fp16'] = f'failed: {e}'[:300]
        if trt == 'int8':
            try:
                kw = {'quantize': 8} if 'quantize' in prec(16) else {'int8': True}
                eng = load_model(weights, rtdetr).export(format='engine', imgsz=imgsz, device=device, data=data_yaml,
                                                         fraction=0.3, **kw)
                m8 = load_model(eng, rtdetr)
                bench(m8, device, {}, 'tensorrt_int8')
                out['tensorrt_int8']['test_mAP50'] = round(float(m8.val(data=data_yaml, split='test', imgsz=imgsz, workers=common.WORKERS,
                                                           batch=1, device=device, verbose=False).box.map50), 4)
            except Exception as e:
                out['tensorrt_int8'] = f'failed: {e}'[:300]
    try:
        out['cpu_device'] = platform.processor() or 'cpu'
        legacy_onnx_export()
        onnx = load_model(weights, rtdetr).export(format='onnx', imgsz=imgsz, simplify=False, dynamic=False)
        bench(load_model(onnx, rtdetr), 'cpu', {}, 'cpu_onnx')
    except Exception as e:
        out['cpu_onnx'] = f'failed: {e}'[:300]
    return out


def add_time_cap(model, hours):
    """Safety net for the 12 h Kaggle limit: stop after the epoch that passes `hours` (epochs stay as set, unlike
    Ultralytics' own `time`, which re-plans the epoch count)."""
    if not hours:
        return
    t0 = time.time()

    def cb(trainer):
        if time.time() - t0 > hours * 3600:
            trainer.stop = True
            print(f'[time cap] {hours} h reached after epoch {trainer.epoch + 1}: stopping', flush=True)
    model.add_callback('on_fit_epoch_end', cb)


def train_once(a, weights, name, epochs, extra):
    model = load_model(weights, a.rtdetr)
    add_time_cap(model, a.max_hours)
    kw = dict(data=a.data, epochs=epochs, imgsz=a.imgsz, batch=a.batch, device=a.device, workers=a.workers,
              seed=a.seed, patience=a.patience, optimizer=a.optimizer, cache=a.cache if a.cache != 'none' else False,
              fraction=a.fraction, project=a.project, name=name, exist_ok=True, plots=True, verbose=True)
    if a.lr0 is not None:
        kw['lr0'] = a.lr0
    if a.time:
        kw['time'] = a.time
    kw.update(extra)
    model.train(**kw)
    return Path(a.project) / name


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--data', required=True)
    p.add_argument('--model', default='yolov8s.pt')
    p.add_argument('--rtdetr', action='store_true')
    p.add_argument('--name', required=True)
    p.add_argument('--project', required=True)
    p.add_argument('--epochs', type=int, default=100)
    p.add_argument('--imgsz', type=int, default=640)
    p.add_argument('--batch', type=int, default=16)
    p.add_argument('--device', default='0')
    p.add_argument('--workers', type=int, default=common.WORKERS)
    p.add_argument('--seed', type=int, default=42)
    p.add_argument('--patience', type=int, default=100)
    p.add_argument('--optimizer', default='SGD', help="SGD by default: Ultralytics 8.4 'auto' picks MuSGD (YOLO26)")
    p.add_argument('--lr0', type=float, default=None)
    p.add_argument('--time', type=float, default=None, help='hours; caps training time (Ultralytics adjusts epochs)')
    p.add_argument('--max-hours', type=float, default=None, help='stop after this many hours (keeps the epoch plan)')
    p.add_argument('--cache', default='none')
    p.add_argument('--fraction', type=float, default=1.0, help='<1 only for the quick test')
    p.add_argument('--hyp', default='{}', help='JSON of extra Ultralytics train arguments (augmentation etc.)')
    p.add_argument('--stages', default='', help='JSON list of {epochs, freeze, hyp}: trained one after another')
    p.add_argument('--labels', default='human+auto', help='which labels were used (written to results.json)')
    p.add_argument('--note', default='')
    p.add_argument('--fps-only', action='store_true')
    p.add_argument('--trt', nargs='?', const='fp16', default='', help='also TensorRT FP16 (--trt) or FP16+INT8 (--trt int8)')
    p.add_argument('--cpu', action='store_true')
    p.add_argument('--test-max', type=int, default=None, help='quick test: score only N test images')
    a = p.parse_args()

    from ultralytics import __version__ as ul_version
    run = Path(a.project) / a.name
    res_file = run / 'results.json'
    if a.fps_only:
        r = json.loads(res_file.read_text())
        r['speed'] = measure_speed(run / 'weights' / 'best.pt', a.data, a.imgsz, a.device, a.rtdetr, trt=a.trt, cpu=a.cpu)
        res_file.write_text(json.dumps(r, indent=2))
        print(a.name, 'speed:', json.dumps(r['speed']))
        return

    hyp = json.loads(a.hyp)
    t0 = time.time()
    stage_runs = []
    if a.stages:
        weights = a.model
        for i, st in enumerate(json.loads(a.stages)):
            extra = dict(hyp, **st.get('hyp', {}))
            if 'freeze' in st:
                extra['freeze'] = st['freeze']
            sname = f'{a.name}_stage{i + 1}'
            rd = train_once(a, weights, sname, st['epochs'], extra)
            weights = rd / 'weights' / 'best.pt'
            stage_runs.append(sname)
        run.mkdir(parents=True, exist_ok=True)
        (run / 'weights').mkdir(exist_ok=True)
        import shutil
        shutil.copy2(weights, run / 'weights' / 'best.pt')
        shutil.copy2(Path(a.project) / stage_runs[-1] / 'weights' / 'last.pt', run / 'weights' / 'last.pt')
        shutil.copy2(Path(a.project) / stage_runs[-1] / 'args.yaml', run / 'args.yaml')
        import pandas as pd
        logs = []
        for i, sname in enumerate(stage_runs):
            df = pd.read_csv(Path(a.project) / sname / 'results.csv')
            df.columns = [c.strip() for c in df.columns]
            df['stage'] = i + 1
            logs.append(df)
        log = pd.concat(logs, ignore_index=True)
        log['epoch'] = range(1, len(log) + 1)
        log.to_csv(run / 'results.csv', index=False)
    else:
        train_once(a, a.model, a.name, a.epochs, hyp)
        import pandas as pd
        log = pd.read_csv(run / 'results.csv')
        log.columns = [c.strip() for c in log.columns]
    train_min = (time.time() - t0) / 60
    key50 = [c for c in log.columns if 'mAP50(B)' in c][0]
    key95 = [c for c in log.columns if 'mAP50-95(B)' in c][0]
    fit = 0.1 * log[key50] + 0.9 * log[key95]

    best = load_model(run / 'weights' / 'best.pt', a.rtdetr)
    try:
        from ultralytics.utils.torch_utils import get_flops, get_num_params
        n_params, gflops = get_num_params(best.model), get_flops(best.model, imgsz=a.imgsz)
    except Exception:
        n_params, gflops = 0, 0
    r = {
        'run': a.name, 'labels': a.labels, 'note': a.note, 'stages': stage_runs,
        'settings': {**yaml.safe_load((run / 'args.yaml').read_text()), 'start_weights': a.model, 'hyp': hyp,
                     'stages': json.loads(a.stages) if a.stages else None},
        'epochs_planned': a.epochs, 'epochs_run': int(len(log)), 'best_epoch': int(log['epoch'].iloc[int(fit.idxmax())]),
        'train_minutes': round(train_min, 1), 'params_M': round(n_params / 1e6, 2), 'GFLOPs': round(float(gflops), 1),
        'val': val_metrics(best, a.data, 'val', a.imgsz, a.device, run),
        'test': val_metrics(best, a.data, 'test', a.imgsz, a.device, run),
        'test_plates': plate_counts(best, a.data, 'test', a.imgsz, a.device, max_images=a.test_max),
        'test_by_condition': condition_metrics(best, a.data, a.imgsz, a.device, run, max_images=a.test_max),
        'versions': {'ultralytics': ul_version, 'torch': torch.__version__, 'python': platform.python_version()},
        'train_device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'peak_gpu_mem_GB': round(torch.cuda.max_memory_reserved() / 2 ** 30, 2) if torch.cuda.is_available() else None,
    }
    res_file.write_text(json.dumps(r, indent=2))
    print('DONE', a.name, '| test:', r['test'], '| plates:', r['test_plates'], f"| {r['train_minutes']} min")


if __name__ == '__main__':
    main()

Writing /kaggle/working/lprkit/det.py


In [7]:
%%writefile /kaggle/working/lprkit/rec.py
"""Character-level YOLO OCR ("detect every Bangla token as an object", paper #2 style): train, test, read plates.

  python rec.py --data rec.yaml --model yolov8n.pt --name chars_v8n --project runs --epochs 100 --imgsz 320
  python rec.py --data rec.yaml --name chars_v8n --project runs --ocr-only        (score the OCR again)

After training: character-box mAP on the recognition test split, then every plate crop of ocr/test.csv is read:
tokens (district, মেট্রো, letter, digits) are joined two ways and scored against the human text:
  rows : words above, digits below (vocab.plate_text)  - our decoder
  xsort: all tokens left to right (what paper #2 describes)
Writes results.json (+ ocr_preds_test.csv with one line per plate for fusion, paper #24).
"""
import argparse
import csv
import json
import os
import platform
import sys
import time
from pathlib import Path

import numpy as np
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import common
import metrics
import vocab
from det import add_time_cap, val_metrics


def read_csv(path):
    with open(path, encoding='utf-8') as f:
        return list(csv.DictReader(f))


def decode(names, xyxy, conf, cls, min_conf=0.25):
    keep = [(names[int(c)], b, float(s)) for b, s, c in zip(xyxy, conf, cls) if s >= min_conf]
    toks = [[n] + list(map(float, b)) for n, b, s in keep]
    text, ok = vocab.plate_text(toks)
    xs = ' '.join(n for n, b, s in sorted(keep, key=lambda k: (k[1][0] + k[1][2]) / 2))
    mean_conf = float(np.mean([s for _, _, s in keep])) if keep else 0.0
    return text, xs, ok, mean_conf


def read_plates(weights, split='test', imgsz=320, device='0', max_n=None, rtdetr=False):
    from ultralytics import YOLO
    root = common.find_dataset()
    rows = read_csv(os.path.join(root, 'ocr', f'{split}.csv'))[:max_n]
    model = YOLO(str(weights))
    names = model.names
    out, ms = [], []
    paths = [os.path.join(root, 'ocr', 'images', split, r['file']) for r in rows]
    for s in range(0, len(paths), 64):
        for r, res in zip(rows[s:s + 64], model.predict(paths[s:s + 64], imgsz=imgsz, conf=0.10, device=device,
                                                         verbose=False)):
            b = res.boxes
            text, xs, ok, mc = decode(names, b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy())
            out.append({'file': r['file'], 'true': r['text'], 'pred': text, 'pred_xsort': xs, 'valid': ok,
                        'conf': round(mc, 4), 'source': r['source']})
    for p in paths[:100]:                                       # batch-1 latency per plate
        res = model.predict(p, imgsz=imgsz, device=device, verbose=False)[0]
        ms.append(sum(res.speed.values()))
    return out, float(np.mean(ms[10:])) if len(ms) > 10 else float(np.mean(ms or [0]))


def ocr_report(preds, ms):
    rep = {'rows_decoder': metrics.score_rows((p['pred'], p['true']) for p in preds),
           'xsort_decoder': metrics.score_rows((p['pred_xsort'], p['true']) for p in preds),
           'valid_format_%': round(100 * np.mean([p['valid'] for p in preds]), 2) if preds else 0,
           'ms_per_plate': round(ms, 2)}
    by_src = {}
    for src in sorted({p['source'] for p in preds}):
        by_src[src] = metrics.score_rows((p['pred'], p['true']) for p in preds if p['source'] == src)
    rep['by_source'] = by_src
    return rep


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--data', required=True)
    p.add_argument('--model', default='yolov8n.pt')
    p.add_argument('--name', required=True)
    p.add_argument('--project', required=True)
    p.add_argument('--epochs', type=int, default=100)
    p.add_argument('--imgsz', type=int, default=320)
    p.add_argument('--batch', type=int, default=32)
    p.add_argument('--device', default='0')
    p.add_argument('--workers', type=int, default=common.WORKERS)
    p.add_argument('--seed', type=int, default=42)
    p.add_argument('--optimizer', default='SGD')
    p.add_argument('--lr0', type=float, default=None)
    p.add_argument('--time', type=float, default=None)
    p.add_argument('--max-hours', type=float, default=None)
    p.add_argument('--fraction', type=float, default=1.0)
    p.add_argument('--hyp', default='{}')
    p.add_argument('--ocr-only', action='store_true')
    p.add_argument('--ocr-max', type=int, default=None)
    p.add_argument('--note', default='')
    a = p.parse_args()

    from ultralytics import YOLO, __version__ as ul_version
    run = Path(a.project) / a.name
    res_file = run / 'results.json'
    if not a.ocr_only:
        hyp = {'fliplr': 0.0, 'mosaic': 0.5, 'degrees': 3.0, 'shear': 2.0, 'scale': 0.3}   # never mirror text
        hyp.update(json.loads(a.hyp))
        model = YOLO(a.model)
        add_time_cap(model, a.max_hours)
        t0 = time.time()
        kw = dict(data=a.data, epochs=a.epochs, imgsz=a.imgsz, batch=a.batch, device=a.device, workers=a.workers,
                  seed=a.seed, optimizer=a.optimizer, fraction=a.fraction, project=a.project, name=a.name,
                  exist_ok=True, plots=True, verbose=True, patience=100, **hyp)
        if a.lr0 is not None:
            kw['lr0'] = a.lr0
        if a.time:
            kw['time'] = a.time
        model.train(**kw)
        import pandas as pd
        log = pd.read_csv(run / 'results.csv')
        log.columns = [c.strip() for c in log.columns]
        best = YOLO(run / 'weights' / 'best.pt')
        r = {'run': a.name, 'kind': 'char-yolo', 'note': a.note,
             'settings': {'start_weights': a.model, 'epochs': a.epochs, 'imgsz': a.imgsz, 'batch': a.batch,
                          'optimizer': a.optimizer, 'hyp': hyp},
             'epochs_planned': a.epochs, 'epochs_run': int(len(log)), 'train_minutes': round((time.time() - t0) / 60, 1),
             'val_boxes': val_metrics(best, a.data, 'val', a.imgsz, a.device, run, batch=32),
             'test_boxes': val_metrics(best, a.data, 'test', a.imgsz, a.device, run, batch=32),
             'versions': {'ultralytics': ul_version, 'torch': torch.__version__, 'python': platform.python_version()},
             'train_device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}
    else:
        r = json.loads(res_file.read_text(encoding='utf-8'))
    preds, ms = read_plates(run / 'weights' / 'best.pt', 'test', a.imgsz, a.device, a.ocr_max)
    r['ocr_test'] = ocr_report(preds, ms)
    root = common.find_dataset()
    r['ocr_test']['by_condition'] = metrics.condition_scores(preds, root)
    r['ocr_test']['extra_tests'] = {}
    for sp in metrics.extra_splits(root):            # real fog crops, synthetic rain copies
        ep, _ = read_plates(run / 'weights' / 'best.pt', sp, a.imgsz, a.device, a.ocr_max)
        r['ocr_test']['extra_tests'][sp] = metrics.score_rows((p['pred'], p['true']) for p in ep)
    vpreds, _ = read_plates(run / 'weights' / 'best.pt', 'val', a.imgsz, a.device, a.ocr_max)
    r['ocr_val'] = metrics.score_rows((p['pred'], p['true']) for p in vpreds)
    for name, rows in (('test', preds), ('val', vpreds)):
        with open(run / f'ocr_preds_{name}.csv', 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader()
            w.writerows(rows)
    res_file.write_text(json.dumps(r, indent=2, ensure_ascii=False), encoding='utf-8')
    print('DONE', a.name, '| boxes:', r.get('test_boxes'), '| OCR:', r['ocr_test']['rows_decoder'])


if __name__ == '__main__':
    main()

Writing /kaggle/working/lprkit/rec.py


In [8]:
%%writefile /kaggle/working/lprkit/seq.py
"""Sequence OCR on plate crops (ocr/{split}.csv): CRNN-CTC or a small ViTSTR, trained from the human plate texts.

  python seq.py --model crnn   --name crnn   --project runs --epochs 60
  python seq.py --model vitstr --name vitstr --project runs --epochs 60
  python seq.py ... --train-extra synth.csv      (extra training rows: file paths + text, e.g. synthetic plates)

Labels are TOKENS (district, মেট্রো, letter, digits; vocab.text_tokens), one class each, so Bangla conjuncts and
vowel signs never have to be spelled. Two-line plates: the crop is cut into top and bottom halves that are placed side
by side ("stack" = rows -> one line, so a left-to-right CTC can read the top row then the bottom row).
Writes results.json + ocr_preds_test.csv (for fusion, paper #24).
"""
import argparse
import csv
import json
import math
import os
import random
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import common
import metrics
import vocab


def load_rows(split, extra=None):
    root = common.find_dataset()
    rows = []
    with open(os.path.join(root, 'ocr', f'{split}.csv'), encoding='utf-8') as f:
        for r in csv.DictReader(f):
            rows.append({'path': os.path.join(root, 'ocr', 'images', split, r['file']), 'text': r['text'],
                         'source': r['source'], 'file': r['file']})
    if extra:
        with open(extra, encoding='utf-8') as f:
            for r in csv.DictReader(f):
                rows.append({'path': r['path'], 'text': r['text'], 'source': r.get('source', 'extra'), 'file': r['path']})
    return rows


def class_list():
    root = common.find_dataset()
    return json.load(open(os.path.join(root, 'vocab.json'), encoding='utf-8'))['classes']


def prep(img, H, W, stack, train, flip=False):
    """grey, optional augment, two-line stacking, aspect-kept resize + pad to H x W, float tensor."""
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    if train:
        h, w = g.shape
        ang = random.uniform(-4, 4)
        M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, random.uniform(0.92, 1.05))
        M[:, 2] += [random.uniform(-0.04, 0.04) * w, random.uniform(-0.05, 0.05) * h]
        g = cv2.warpAffine(g, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
        g = np.clip(g.astype(np.float32) * random.uniform(0.6, 1.3) + random.uniform(-30, 30), 0, 255).astype(np.uint8)
        if random.random() < 0.3:
            k = random.choice([3, 5]); g = cv2.GaussianBlur(g, (k, k), 0)
        if random.random() < 0.3:
            g = np.clip(g + np.random.normal(0, random.uniform(3, 12), g.shape), 0, 255).astype(np.uint8)
        if random.random() < 0.2:
            s = random.uniform(0.35, 0.7)
            g = cv2.resize(cv2.resize(g, None, fx=s, fy=s), (g.shape[1], g.shape[0]))
        if flip:
            g = g[:, ::-1]
    h, w = g.shape
    if stack and w / max(1, h) < 2.6:                 # two-line plate -> one line
        top, bot = g[: h // 2], g[h // 2:]
        g = np.concatenate([top, cv2.resize(bot, (w, top.shape[0]))], 1)
        h, w = g.shape
    s = min(H / h, W / w)
    nh, nw = max(1, int(h * s)), max(1, int(w * s))
    canvas = np.full((H, W), 127, np.uint8)
    y0, x0 = (H - nh) // 2, (W - nw) // 2
    canvas[y0:y0 + nh, x0:x0 + nw] = cv2.resize(g, (nw, nh), interpolation=cv2.INTER_AREA)
    return torch.from_numpy(canvas).float().div(255).sub(0.5).div(0.5)[None]


class DS(torch.utils.data.Dataset):
    def __init__(self, rows, cls, H, W, stack, train, flip=False):
        self.rows, self.idx, self.H, self.W, self.stack, self.train, self.flip = rows, {c: i + 1 for i, c in enumerate(cls)}, H, W, stack, train, flip

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        img = cv2.imread(r['path'])
        if img is None:
            img = np.zeros((40, 120, 3), np.uint8)
        x = prep(img, self.H, self.W, self.stack, self.train, self.flip and random.random() < 0.5)
        y = [self.idx[t] for t in vocab.text_tokens(r['text']) if t in self.idx]
        return x, torch.tensor(y, dtype=torch.long), i


def collate(b):
    xs, ys, ids = zip(*b)
    return torch.stack(xs), list(ys), list(ids)


class CRNN(nn.Module):
    """VGG-style CNN -> height collapsed -> 2-layer BiLSTM -> per-column classes (CTC, class 0 = blank)."""
    def __init__(self, n_cls, H=48):
        super().__init__()
        def blk(i, o, pool):
            return [nn.Conv2d(i, o, 3, 1, 1), nn.BatchNorm2d(o), nn.ReLU(True)] + ([nn.MaxPool2d(pool)] if pool else [])
        self.cnn = nn.Sequential(*blk(1, 64, (2, 2)), *blk(64, 128, (2, 2)), *blk(128, 256, None),
                                 *blk(256, 256, (2, 1)), *blk(256, 384, None), *blk(384, 384, (2, 1)))
        self.rnn = nn.LSTM(384 * (H // 16), 256, num_layers=2, bidirectional=True, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(512, n_cls + 1)

    def forward(self, x):
        f = self.cnn(x)                               # B, C, H/16, W/4
        b, c, h, w = f.shape
        f = f.permute(0, 3, 1, 2).reshape(b, w, c * h)
        return self.fc(self.rnn(f)[0])                # B, T, n+1


class ViTSTR(nn.Module):
    """ViTSTR-style: a small pretrained ViT (timm) reads the image; the first L output tokens are classified
    (position i -> token i of the plate, 0 = end/padding)."""
    def __init__(self, n_cls, H, W, L=12, name='vit_tiny_patch16_224'):
        super().__init__()
        import timm
        self.vit = timm.create_model(name, pretrained=True, img_size=(H, W), in_chans=1, num_classes=0)
        self.L = L
        self.head = nn.Linear(self.vit.num_features, n_cls + 1)

    def forward(self, x):
        f = self.vit.forward_features(x)              # B, 1+N, D
        return self.head(f[:, 1:self.L + 1])          # B, L, n+1


class SlotCNN(nn.Module):
    """fast-plate-ocr style (papers #18, #24): plain CNN on a 64x128 grey crop, then L fixed character slots, each a
    softmax over the alphabet (0 = empty slot). No CTC, no attention: very fast."""
    def __init__(self, n_cls, L=9, width=1.0):
        super().__init__()
        c = lambda x: int(x * width)
        def blk(i, o):
            return [nn.Conv2d(i, o, 3, 1, 1), nn.BatchNorm2d(o), nn.ReLU(True)]
        self.f = nn.Sequential(*blk(1, c(32)), nn.MaxPool2d(2), *blk(c(32), c(64)), nn.MaxPool2d(2),
                               *blk(c(64), c(128)), *blk(c(128), c(128)), nn.MaxPool2d(2),
                               *blk(c(128), c(256)), *blk(c(256), c(256)), nn.MaxPool2d(2),
                               nn.Conv2d(c(256), c(512), 1), nn.ReLU(True))
        self.L, self.n = L, n_cls + 1
        self.head = nn.Conv2d(c(512), L * (n_cls + 1), 1)

    def forward(self, x):
        y = self.head(self.f(x))                         # B, L*(n+1), h, w
        y = y.amax(dim=(2, 3))                           # global max pool (fast-plate-ocr)
        return y.view(x.shape[0], self.L, self.n)


def ctc_decode(logits):
    best = logits.argmax(-1).cpu().numpy()
    out, conf = [], []
    probs = logits.softmax(-1).max(-1).values.cpu().numpy()
    for seq, pr in zip(best, probs):
        toks, cs, prev = [], [], 0
        for k, (c, p) in enumerate(zip(seq, pr)):
            if c != 0 and c != prev:
                toks.append(int(c)); cs.append(float(p))
            prev = c
        out.append(toks); conf.append(float(np.mean(cs)) if cs else 0.0)
    return out, conf


def fixed_decode(logits):
    best = logits.argmax(-1).cpu().numpy()
    probs = logits.softmax(-1).max(-1).values.cpu().numpy()
    out, conf = [], []
    for seq, pr in zip(best, probs):
        toks, cs = [], []
        for c, p in zip(seq, pr):
            if c == 0:
                break
            toks.append(int(c)); cs.append(float(p))
        out.append(toks); conf.append(float(np.mean(cs)) if cs else 0.0)
    return out, conf


def run_epoch(model, dl, opt, sched, kind, device, scaler, train):
    model.train(train)
    tot, n = 0.0, 0
    ctc = nn.CTCLoss(blank=0, zero_infinity=True)
    for x, ys, _ in dl:
        x = x.to(device)
        with torch.autocast('cuda', enabled=device.type == 'cuda'):
            out = model(x)
        out = out.float()
        if kind == 'crnn':
            lp = out.log_softmax(-1).permute(1, 0, 2)
            tgt = torch.cat([y for y in ys]).to(device)
            loss = ctc(lp, tgt, torch.full((x.shape[0],), lp.shape[0], dtype=torch.long),
                       torch.tensor([len(y) for y in ys], dtype=torch.long))
        else:
            L = out.shape[1]
            tgt = torch.zeros((x.shape[0], L), dtype=torch.long)
            for i, y in enumerate(ys):
                tgt[i, :min(L, len(y))] = y[:L]
            loss = F.cross_entropy(out.reshape(-1, out.shape[-1]), tgt.to(device).reshape(-1), label_smoothing=0.05)
        if train:
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update(); sched.step()
        tot += float(loss.detach()) * x.shape[0]; n += x.shape[0]
    return tot / max(1, n)


@torch.no_grad()
def predict(model, rows, cls, kind, a, device):
    model.eval()
    ds = DS(rows, cls, a.H, a.W, a.stack, False)
    dl = torch.utils.data.DataLoader(ds, batch_size=64, shuffle=False, num_workers=min(2, common.WORKERS), collate_fn=collate)
    preds = []
    for x, ys, ids in dl:
        out = model(x.to(device)).float()
        toks, conf = ctc_decode(out) if kind == 'crnn' else fixed_decode(out)
        for i, t, c in zip(ids, toks, conf):
            text = vocab.tokens_to_text([cls[k - 1] for k in t])
            r = rows[i]
            preds.append({'file': r['file'], 'true': r['text'], 'pred': text, 'conf': round(c, 4), 'source': r['source']})
    # batch-1 latency
    ms = []
    for r in rows[:120]:
        x = ds[rows.index(r)][0][None].to(device)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(x)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        ms.append(1000 * (time.perf_counter() - t0))
    return preds, float(np.mean(ms[20:])) if len(ms) > 20 else 0.0


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--model', default='crnn', choices=['crnn', 'vitstr', 'slot'])
    p.add_argument('--name', required=True)
    p.add_argument('--project', required=True)
    p.add_argument('--epochs', type=int, default=60)
    p.add_argument('--batch', type=int, default=64)
    p.add_argument('--lr', type=float, default=None)
    p.add_argument('--H', type=int, default=48)
    p.add_argument('--W', type=int, default=256)
    p.add_argument('--no-stack', dest='stack', action='store_false')
    p.add_argument('--flip', action='store_true', help='random horizontal flip (only to show why it is wrong)')
    p.add_argument('--train-extra', default='', help='csv with path,text[,source] rows added to training')
    p.add_argument('--real-fraction', type=float, default=1.0, help='use only this fraction of the real train rows')
    p.add_argument('--seed', type=int, default=42)
    p.add_argument('--max-test', type=int, default=None)
    p.add_argument('--note', default='')
    a = p.parse_args()
    random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    run = Path(a.project) / a.name
    run.mkdir(parents=True, exist_ok=True)
    cls = class_list()
    tr = load_rows('train')
    if a.real_fraction < 1:
        random.Random(a.seed).shuffle(tr)
        tr = tr[:max(1, int(len(tr) * a.real_fraction))]
    n_real = len(tr)
    if a.train_extra:
        tr += load_rows('train', a.train_extra)[len(load_rows('train')):]
    va, te = load_rows('val'), load_rows('test')[:a.max_test]
    if a.model == 'crnn':
        model, lr = CRNN(len(cls), a.H), a.lr or 1e-3
    elif a.model == 'slot':
        a.H, a.W = (64, 128) if a.H == 48 else (a.H, a.W)
        a.stack = False                                   # the CNN sees the whole two-line plate
        model, lr = SlotCNN(len(cls)), a.lr or 1e-3
    else:
        a.H, a.W = (32, 256) if a.H == 48 else (a.H, a.W)
        model, lr = ViTSTR(len(cls), a.H, a.W), a.lr or 3e-4
    model.to(device)
    dl = torch.utils.data.DataLoader(DS(tr, cls, a.H, a.W, a.stack, True, a.flip), batch_size=a.batch, shuffle=True,
                                     num_workers=common.WORKERS, collate_fn=collate, drop_last=True)
    dv = torch.utils.data.DataLoader(DS(va, cls, a.H, a.W, a.stack, False), batch_size=a.batch, num_workers=min(2, common.WORKERS),
                                     collate_fn=collate)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=a.epochs * max(1, len(dl)), pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=device.type == 'cuda')
    t0, best, log = time.time(), 1e9, []
    for ep in range(a.epochs):
        tl = run_epoch(model, dl, opt, sched, a.model, device, scaler, True)
        with torch.no_grad():
            vl = run_epoch(model, dv, None, None, a.model, device, scaler, False)
        log.append({'epoch': ep + 1, 'train_loss': round(tl, 4), 'val_loss': round(vl, 4)})
        if vl < best:
            best = vl
            torch.save(model.state_dict(), run / 'best.pt')
        print(f'epoch {ep + 1}/{a.epochs} train {tl:.4f} val {vl:.4f}', flush=True)
    train_min = (time.time() - t0) / 60
    model.load_state_dict(torch.load(run / 'best.pt', map_location=device))
    preds, ms = predict(model, te, cls, a.model, a, device)
    vpred, _ = predict(model, va, cls, a.model, a, device)
    r = {'run': a.name, 'kind': a.model, 'note': a.note,
         'settings': {'epochs': a.epochs, 'batch': a.batch, 'lr': lr, 'input': [a.H, a.W], 'stack_two_lines': a.stack,
                      'flip': a.flip, 'train_rows': len(tr), 'real_rows': n_real, 'extra': a.train_extra,
                      'optimizer': 'AdamW + OneCycle', 'seed': a.seed},
         'train_minutes': round(train_min, 1), 'params_M': round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
         'val_ocr': metrics.score_rows((p['pred'], p['true']) for p in vpred),
         'ocr_test': {'rows_decoder': metrics.score_rows((p['pred'], p['true']) for p in preds),
                      'ms_per_plate': round(ms, 2),
                      'by_source': {s: metrics.score_rows((p['pred'], p['true']) for p in preds if p['source'] == s)
                                    for s in sorted({p['source'] for p in preds})}},
         'log': log, 'env': common.env_info()}
    root = common.find_dataset()
    r['ocr_test']['by_condition'] = metrics.condition_scores(preds, root)
    r['ocr_test']['extra_tests'] = {}
    for sp in metrics.extra_splits(root):            # real fog crops, synthetic rain copies
        ep, _ = predict(model, load_rows(sp)[:a.max_test], cls, a.model, a, device)
        r['ocr_test']['extra_tests'][sp] = metrics.score_rows((p['pred'], p['true']) for p in ep)
    common.save_json(r, run / 'results.json')
    for name, rows in (('test', preds), ('val', vpred)):
        with open(run / f'ocr_preds_{name}.csv', 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader(); w.writerows(rows)
    torch.save({'state_dict': model.state_dict(), 'kind': a.model, 'classes': cls, 'H': a.H, 'W': a.W,
                'stack': a.stack}, run / 'model_full.pt')
    print('DONE', a.name, r['ocr_test']['rows_decoder'], f'{train_min:.1f} min')


if __name__ == '__main__':
    main()

Writing /kaggle/working/lprkit/seq.py


In [9]:
%%writefile /kaggle/working/lprkit/ved.py
"""Transformer OCR: Hugging Face VisionEncoderDecoder (image encoder + text decoder) on plate crops (ocr/{split}.csv).

  python ved.py --encoder google/vit-base-patch16-384 --decoder sagorsarker/bangla-bert-base --name vit_bbert ...
  python ved.py --encoder google/vit-base-patch16-384 --decoder xlm-roberta-base --name vit_xlmr ...
  python ved.py --trocr microsoft/trocr-base-stage1 --name trocr ...

Paper #1 settings (Table I): dropout 0.3 (encoder, decoder, attention), max length 20, beam 3, no-repeat n-gram 0,
length penalty 1.0, weight decay 0.01, fp16, batch 6. Paper #32: random flip + rotation ±5°, AdamW (flip is an option
here only to measure its effect). Images are padded to a square with grey (default) or stretched (--stretch).
Writes results.json + ocr_preds_test.csv and ocr_preds_val.csv (for fusion, paper #24).
"""
import argparse
import csv
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageOps

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import common
import metrics
import vocab
from seq import load_rows


class DS(torch.utils.data.Dataset):
    def __init__(self, rows, processor, tokenizer, max_len, train, stretch, flip):
        self.rows, self.proc, self.tok, self.max_len, self.train, self.stretch, self.flip = rows, processor, tokenizer, max_len, train, stretch, flip

    def __len__(self):
        return len(self.rows)

    def image(self, path):
        im = Image.open(path).convert('RGB')
        if self.train:
            im = im.rotate(random.uniform(-5, 5), resample=Image.BILINEAR, expand=False, fillcolor=(127, 127, 127))
            if self.flip and random.random() < 0.5:
                im = ImageOps.mirror(im)
        if not self.stretch:
            w, h = im.size
            s = max(w, h)
            canvas = Image.new('RGB', (s, s), (127, 127, 127))
            canvas.paste(im, ((s - w) // 2, (s - h) // 2))
            im = canvas
        return im

    def __getitem__(self, i):
        r = self.rows[i]
        px = self.proc(images=self.image(r['path']), return_tensors='pt').pixel_values[0]
        text = vocab.N(r['text'])
        ids = self.tok(text, padding='max_length', max_length=self.max_len, truncation=True).input_ids
        labels = torch.tensor([t if t != self.tok.pad_token_id else -100 for t in ids])
        return px, labels, i


def build(a):
    from transformers import (AutoImageProcessor, AutoTokenizer, TrOCRProcessor, VisionEncoderDecoderModel)
    if a.trocr:
        proc = TrOCRProcessor.from_pretrained(a.trocr)
        model = VisionEncoderDecoderModel.from_pretrained(a.trocr)
        image_proc, tok = proc.image_processor, proc.tokenizer
        model.config.decoder_start_token_id = model.config.decoder_start_token_id or tok.cls_token_id or tok.bos_token_id
    else:
        image_proc = AutoImageProcessor.from_pretrained(a.encoder)
        kw = {}
        if 'bert' in a.decoder.lower() and 'roberta' not in a.decoder.lower():
            kw = {'do_lower_case': False, 'strip_accents': False}   # BERT's accent stripping deletes Bangla vowel signs
        tok = AutoTokenizer.from_pretrained(a.decoder, **kw)
        model = VisionEncoderDecoderModel.from_encoder_decoder_pretrained(
            a.encoder, a.decoder,
            encoder_hidden_dropout_prob=a.dropout, encoder_attention_probs_dropout_prob=a.dropout,
            decoder_hidden_dropout_prob=a.dropout, decoder_attention_probs_dropout_prob=a.dropout)
        model.config.decoder.is_decoder = True
        model.config.decoder.add_cross_attention = True
        model.config.decoder_start_token_id = tok.cls_token_id if tok.cls_token_id is not None else tok.bos_token_id
        model.config.eos_token_id = tok.sep_token_id if tok.sep_token_id is not None else tok.eos_token_id
    model.config.pad_token_id = tok.pad_token_id
    model.config.vocab_size = model.config.decoder.vocab_size
    gc = model.generation_config
    gc.decoder_start_token_id = model.config.decoder_start_token_id
    gc.eos_token_id = model.config.eos_token_id
    gc.pad_token_id = tok.pad_token_id
    gc.max_length, gc.num_beams, gc.no_repeat_ngram_size, gc.length_penalty, gc.early_stopping = a.max_len, a.beams, 0, 1.0, True
    return model, image_proc, tok


@torch.no_grad()
def predict(model, tok, ds, device, bs=16):
    model.eval()
    dl = torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=False, num_workers=min(2, common.WORKERS))
    out = []
    for px, _, ids in dl:
        with torch.autocast('cuda', enabled=device.type == 'cuda'):
            gen = model.generate(px.to(device), output_scores=True, return_dict_in_generate=True)
        texts = tok.batch_decode(gen.sequences, skip_special_tokens=True)
        scores = gen.sequences_scores.exp().tolist() if getattr(gen, 'sequences_scores', None) is not None else [0.0] * len(texts)
        for i, t, s in zip(ids.tolist(), texts, scores):
            r = ds.rows[i]
            out.append({'file': r['file'], 'true': r['text'], 'pred': vocab.N(t.replace(' ##', '').strip()),
                        'conf': round(float(s), 4), 'source': r['source']})
    ms = []
    for i in range(min(60, len(ds))):
        px = ds[i][0][None].to(device)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.autocast('cuda', enabled=device.type == 'cuda'):
            model.generate(px)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        ms.append(1000 * (time.perf_counter() - t0))
    return out, float(np.mean(ms[10:])) if len(ms) > 10 else 0.0


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--encoder', default='google/vit-base-patch16-384')
    p.add_argument('--decoder', default='sagorsarker/bangla-bert-base')
    p.add_argument('--trocr', default='', help='a full TrOCR checkpoint instead of encoder+decoder')
    p.add_argument('--name', required=True)
    p.add_argument('--project', required=True)
    p.add_argument('--epochs', type=int, default=10)
    p.add_argument('--batch', type=int, default=6)
    p.add_argument('--accum', type=int, default=1)
    p.add_argument('--lr', type=float, default=5e-5)
    p.add_argument('--wd', type=float, default=0.01)
    p.add_argument('--dropout', type=float, default=0.3)
    p.add_argument('--max-len', type=int, default=20)
    p.add_argument('--beams', type=int, default=3)
    p.add_argument('--stretch', action='store_true')
    p.add_argument('--flip', action='store_true')
    p.add_argument('--max-train', type=int, default=None)
    p.add_argument('--max-test', type=int, default=None)
    p.add_argument('--seed', type=int, default=42)
    p.add_argument('--note', default='')
    a = p.parse_args()
    random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    run = Path(a.project) / a.name
    run.mkdir(parents=True, exist_ok=True)
    model, proc, tok = build(a)
    model.to(device)
    tr, va, te = load_rows('train')[:a.max_train], load_rows('val')[:a.max_test], load_rows('test')[:a.max_test]
    ds_tr = DS(tr, proc, tok, a.max_len, True, a.stretch, a.flip)
    dl = torch.utils.data.DataLoader(ds_tr, batch_size=a.batch, shuffle=True, num_workers=common.WORKERS, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=a.wd)
    steps = a.epochs * len(dl) // a.accum
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 300) * max(0.0, 1 - s / max(1, steps)))
    scaler = torch.amp.GradScaler(enabled=device.type == 'cuda')
    t0, log, step = time.time(), [], 0
    for ep in range(a.epochs):
        model.train()
        tot = n = 0
        for k, (px, labels, _) in enumerate(dl):
            with torch.autocast('cuda', enabled=device.type == 'cuda'):
                loss = model(pixel_values=px.to(device), labels=labels.to(device)).loss / a.accum
            scaler.scale(loss).backward()
            if (k + 1) % a.accum == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); step += 1
            tot += float(loss) * a.accum * px.shape[0]; n += px.shape[0]
        vpred, _ = predict(model, tok, DS(va[:300], proc, tok, a.max_len, False, a.stretch, False), device)
        vs = metrics.score_rows((x['pred'], x['true']) for x in vpred)
        log.append({'epoch': ep + 1, 'train_loss': round(tot / max(1, n), 4), 'val_CER_%': vs['CER_%'],
                    'val_exact_%': vs['plate_exact_%']})
        print(log[-1], flush=True)
        if not any(l['val_CER_%'] < vs['CER_%'] for l in log[:-1]):
            model.save_pretrained(run / 'best'); tok.save_pretrained(run / 'best')
    train_min = (time.time() - t0) / 60
    from transformers import VisionEncoderDecoderModel
    model = VisionEncoderDecoderModel.from_pretrained(run / 'best').to(device)
    preds, ms = predict(model, tok, DS(te, proc, tok, a.max_len, False, a.stretch, False), device)
    vpred, _ = predict(model, tok, DS(va, proc, tok, a.max_len, False, a.stretch, False), device)
    r = {'run': a.name, 'kind': 'vision-encoder-decoder', 'note': a.note,
         'settings': {'encoder': a.encoder if not a.trocr else a.trocr, 'decoder': a.decoder if not a.trocr else a.trocr,
                      'epochs': a.epochs, 'batch': a.batch * a.accum, 'lr': a.lr, 'weight_decay': a.wd,
                      'dropout': a.dropout, 'max_len': a.max_len, 'beams': a.beams, 'pad_to_square': not a.stretch,
                      'flip': a.flip, 'steps': step, 'train_rows': len(tr)},
         'train_minutes': round(train_min, 1), 'params_M': round(sum(p.numel() for p in model.parameters()) / 1e6, 1),
         'val_ocr': metrics.score_rows((x['pred'], x['true']) for x in vpred),
         'ocr_test': {'rows_decoder': metrics.score_rows((x['pred'], x['true']) for x in preds), 'ms_per_plate': round(ms, 1),
                      'by_source': {s: metrics.score_rows((x['pred'], x['true']) for x in preds if x['source'] == s)
                                    for s in sorted({x['source'] for x in preds})}},
         'log': log, 'env': common.env_info()}
    root = common.find_dataset()
    r['ocr_test']['by_condition'] = metrics.condition_scores(preds, root)
    r['ocr_test']['extra_tests'] = {}
    for sp in metrics.extra_splits(root):            # real fog crops, synthetic rain copies
        ep, _ = predict(model, tok, DS(load_rows(sp)[:a.max_test], proc, tok, a.max_len, False, a.stretch, False), device)
        r['ocr_test']['extra_tests'][sp] = metrics.score_rows((x['pred'], x['true']) for x in ep)
    common.save_json(r, run / 'results.json')
    for name, rows in (('test', preds), ('val', vpred)):
        with open(run / f'ocr_preds_{name}.csv', 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader(); w.writerows(rows)
    print('DONE', a.name, r['ocr_test']['rows_decoder'], f'{train_min:.1f} min')


if __name__ == '__main__':
    main()

Writing /kaggle/working/lprkit/ved.py


In [10]:
%%writefile /kaggle/working/lprkit/p08_train.py
"""Paper #8 (Afrin et al., ICCIT 2023, arXiv:2312.10701): the CNN and ResNet-50 character classifiers,
ported from the authors' Keras notebooks to PyTorch so they run on Kaggle (one GPU per job).

  python p08_train.py --task paper --model cnn      --data Data          --out runs/cnn_s42
  python p08_train.py --task paper --model resnet50 --data /tmp/p08_chars --out runs/resnet50 --max-hours 3
  python p08_train.py --fps-only --out runs/cnn_s42          # measure speed on an idle GPU

--task paper : a folder with training_data/, val_data/, testing_data/ and one sub-folder per class: the authors' own
               Data.zip (17 classes) or our token crops written by p08_chars.py (one folder per Bangla token)
--task team  : a split.csv (columns path,label,split,group); labels are the dataset's folders
"""
import argparse
import csv
import json
import platform
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from torchvision import models
from torchvision.transforms import v2 as T

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}

# Settings taken from the paper (page 4-5) and the authors' notebooks (Code/CNN.ipynb, Code/resnet50.ipynb).
# imgsz: the paper text says 32x32, but the code trains the CNN on 28x28 and ResNet-50 on 150x150.
# lr: CNN uses Adam(1e-4); ResNet-50 is compiled with optimizer='adam', i.e. the Keras default 1e-3.
PAPER = {
    "cnn": dict(imgsz=28, epochs=40, batch=1, lr=1e-4),
    "resnet50": dict(imgsz=150, epochs=30, batch=1, lr=1e-3),
}


def get_args():
    p = argparse.ArgumentParser()
    p.add_argument("--task", choices=["paper", "team"])
    p.add_argument("--model", choices=["cnn", "resnet50"])
    p.add_argument("--data", help="paper: folder holding training_data/val_data/testing_data; team: split.csv")
    p.add_argument("--root", default="", help="team: folder that the paths in split.csv are relative to")
    p.add_argument("--out", required=True)
    p.add_argument("--epochs", type=int)
    p.add_argument("--batch", type=int)
    p.add_argument("--lr", type=float)
    p.add_argument("--imgsz", type=int)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--imagenet-norm", action="store_true", help="ResNet-50 only. The authors did NOT normalise (off = faithful).")
    p.add_argument("--no-aug", action="store_true")
    p.add_argument("--limit", type=int, default=0, help="debug: use only N images per split")
    p.add_argument("--max-hours", type=float, default=None, help="stop after the epoch that passes this cap")
    p.add_argument("--note", default="")
    p.add_argument("--fps-only", action="store_true", help="load <out>/best.pt and measure speed only")
    a = p.parse_args()
    if not a.fps_only:
        if not (a.task and a.model and a.data):
            p.error("--task, --model and --data are required for training")
        for k, v in PAPER[a.model].items():
            if getattr(a, k) is None:
                setattr(a, k, v)
    return a


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ---------------------------------------------------------------- data
def list_paper_split(root):
    root = Path(root)
    dirs = {"train": "training_data", "val": "val_data", "test": "testing_data"}
    # Keras flow_from_directory sorts class folders the same way, so class ids match the authors' order.
    # Classes are taken from all three splits: a rare token can have test crops but no training crop (our data).
    classes = sorted({d.name for sub in dirs.values() if (root / sub).is_dir()
                      for d in (root / sub).iterdir() if d.is_dir()})
    items = {}
    for split, sub in dirs.items():
        items[split] = [(str(f), classes.index(c.name))
                        for c in sorted((root / sub).iterdir()) if c.is_dir()
                        for f in sorted(c.iterdir()) if f.suffix.lower() in IMG_EXT]
    return items, classes


def list_team_split(csv_path, root):
    with open(csv_path, newline="", encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))
    classes = sorted({r["label"] for r in rows})
    root = Path(root) if root else Path(csv_path).parent
    items = {s: [(str(root / r["path"]), classes.index(r["label"])) for r in rows if r["split"] == s]
             for s in ("train", "val", "test")}
    return items, classes


def load_images(pairs, imgsz):
    """Decode and resize every image once, keep them in RAM as uint8 (N,3,H,W)."""
    x = torch.empty((len(pairs), 3, imgsz, imgsz), dtype=torch.uint8)
    for i, (path, _) in enumerate(pairs):
        im = Image.open(path).convert("RGB").resize((imgsz, imgsz), Image.BILINEAR)
        x[i] = torch.from_numpy(np.asarray(im).copy()).permute(2, 0, 1)
    y = torch.tensor([lbl for _, lbl in pairs], dtype=torch.long)
    return x, y


def train_augment(model_name):
    if model_name == "cnn":   # ImageDataGenerator(width_shift_range=0.1, height_shift_range=0.1)
        return T.RandomAffine(degrees=0, translate=(0.1, 0.1))
    # ImageDataGenerator(shear_range=0.2, zoom_range=0.2, horizontal_flip=True); Keras shear is in degrees
    return T.Compose([T.RandomAffine(degrees=0, shear=0.2, scale=(0.8, 1.2)), T.RandomHorizontalFlip()])


class TensorSet(torch.utils.data.Dataset):
    def __init__(self, x, y, aug=None):
        self.x, self.y, self.aug = x, y, aug

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        img = self.x[i]
        if self.aug is not None:
            img = self.aug(img)
        return img.float() / 255.0, self.y[i]   # rescale=1./255, as in the paper's code


# ---------------------------------------------------------------- models
class PaperCNN(nn.Module):
    """Code/CNN.ipynb: 4 'same' convs (22,16,8,4 kernels) -> maxpool 4 -> 512 -> dropout 0.4 -> 128 -> classes."""

    def __init__(self, n_classes, imgsz):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 22, padding="same"), nn.ReLU(),
            nn.Conv2d(16, 32, 16, padding="same"), nn.ReLU(),
            nn.Conv2d(32, 64, 8, padding="same"), nn.ReLU(),
            nn.Conv2d(64, 128, 4, padding="same"), nn.ReLU(),
            nn.MaxPool2d(4),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * (imgsz // 4) ** 2, 512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 128), nn.ReLU(),
            nn.Linear(128, n_classes),
        )
        for m in self.modules():   # Keras defaults: glorot_uniform weights, zero bias
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.head(self.features(x))


class PaperResNet50(nn.Module):
    """Code/resnet50.ipynb: ImageNet ResNet-50 without top, all layers frozen, Flatten -> Dense(classes)."""

    def __init__(self, n_classes, imgsz, imagenet_norm=False, pretrained=True):
        super().__init__()
        weights = models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        net = models.resnet50(weights=weights)
        self.backbone = nn.Sequential(*list(net.children())[:-2])
        for p in self.backbone.parameters():
            p.requires_grad = False
        with torch.no_grad():
            feat = self.backbone.eval()(torch.zeros(1, 3, imgsz, imgsz)).numel()
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(feat, n_classes))
        self.imagenet_norm = imagenet_norm
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()   # frozen Keras layers also run BatchNorm in inference mode
        return self

    def forward(self, x):
        if self.imagenet_norm:
            x = (x - self.mean) / self.std
        with torch.no_grad():
            f = self.backbone(x)
        return self.fc(f)


def build_model(name, n_classes, imgsz, imagenet_norm=False, pretrained=True):
    if name == "cnn":
        return PaperCNN(n_classes, imgsz)
    return PaperResNet50(n_classes, imgsz, imagenet_norm, pretrained)


# ---------------------------------------------------------------- train / eval
def run_epoch(model, loader, device, opt=None):
    train = opt is not None
    model.train(train)
    loss_fn = nn.CrossEntropyLoss()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            out = model(xb)
            loss = loss_fn(out, yb)
            if train:
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
            loss_sum += loss.item() * len(yb)
            correct += (out.argmax(1) == yb).sum().item()
            total += len(yb)
    return loss_sum / total, correct / total


@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    preds, gts = [], []
    for xb, yb in loader:
        preds.append(model(xb.to(device)).argmax(1).cpu())
        gts.append(yb)
    return torch.cat(gts).numpy(), torch.cat(preds).numpy()


def metrics(y, p, classes):
    pm, rm, fm, _ = precision_recall_fscore_support(y, p, average="macro", zero_division=0)
    pw, rw, fw, _ = precision_recall_fscore_support(y, p, average="weighted", zero_division=0)
    return {
        "accuracy": accuracy_score(y, p),
        "precision_macro": pm, "recall_macro": rm, "f1_macro": fm,
        "precision_weighted": pw, "recall_weighted": rw, "f1_weighted": fw,
        "report": classification_report(y, p, labels=list(range(len(classes))), target_names=classes, zero_division=0),
        "confusion": confusion_matrix(y, p, labels=list(range(len(classes)))).tolist(),
    }


@torch.no_grad()
def measure_fps(model, imgsz, device, n=300, warmup=50):
    """Model-only speed at batch 1 (the image is already resized and on the GPU)."""
    model.eval()
    x = torch.rand(1, 3, imgsz, imgsz, device=device)
    for _ in range(warmup):
        model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(n):
        model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    dt = time.perf_counter() - t
    return n / dt, 1000 * dt / n


def fps_only(a):
    out = Path(a.out)
    res = json.loads((out / "results.json").read_text())
    s = res["settings"]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(s["model"], len(res["classes"]), s["imgsz"], s["imagenet_norm"], pretrained=False).to(device)
    model.load_state_dict(torch.load(out / "best.pt", map_location=device))
    fps, ms = measure_fps(model, s["imgsz"], device)
    res["speed"] = {"fps_bs1": round(fps, 1), "ms_per_image": round(ms, 2), "device": gpu_name(device),
                    "note": "model only, batch 1, FP32, measured alone on an idle GPU"}
    (out / "results.json").write_text(json.dumps(res, indent=2))
    print(f"{out.name}: {fps:.1f} img/s ({ms:.2f} ms) on {gpu_name(device)}")


def gpu_name(device):
    return torch.cuda.get_device_name(device) if device.type == "cuda" else platform.processor() or "CPU"


def main():
    a = get_args()
    if a.fps_only:
        return fps_only(a)
    seed_everything(a.seed)
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    items, classes = list_paper_split(a.data) if a.task == "paper" else list_team_split(a.data, a.root)
    if a.limit:
        items = {k: random.Random(a.seed).sample(v, min(a.limit, len(v))) for k, v in items.items()}
    print(f"[{out.name}] classes={len(classes)} train={len(items['train'])} val={len(items['val'])} "
          f"test={len(items['test'])} device={gpu_name(device)}", flush=True)

    t0 = time.time()
    data = {k: load_images(v, a.imgsz) for k, v in items.items()}
    print(f"[{out.name}] images loaded in {time.time() - t0:.0f}s", flush=True)
    aug = None if a.no_aug else train_augment(a.model)
    g = torch.Generator().manual_seed(a.seed)
    loaders = {
        "train": torch.utils.data.DataLoader(TensorSet(*data["train"], aug), batch_size=a.batch, shuffle=True, generator=g),
        "val": torch.utils.data.DataLoader(TensorSet(*data["val"]), batch_size=64),
        "test": torch.utils.data.DataLoader(TensorSet(*data["test"]), batch_size=64),
    }

    model = build_model(a.model, len(classes), a.imgsz, a.imagenet_norm).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=a.lr, eps=1e-7)

    log = open(out / "log.csv", "w", newline="")
    w = csv.writer(log)
    w.writerow(["epoch", "train_loss", "train_acc", "val_loss", "val_acc", "epoch_sec"])
    best_acc, best_epoch, epochs_run = -1.0, 0, 0
    t_train = time.time()
    for ep in range(1, a.epochs + 1):
        te = time.time()
        tl, ta = run_epoch(model, loaders["train"], device, opt)
        vl, va = run_epoch(model, loaders["val"], device)
        sec = time.time() - te
        w.writerow([ep, f"{tl:.4f}", f"{ta:.4f}", f"{vl:.4f}", f"{va:.4f}", f"{sec:.1f}"])
        log.flush()
        if va > best_acc:   # keep the epoch with the best VALIDATION accuracy (never pick by test)
            best_acc, best_epoch = va, ep
            torch.save(model.state_dict(), out / "best.pt")
        print(f"[{out.name}] ep {ep:3d}/{a.epochs} loss {tl:.4f} acc {ta:.4f} | val loss {vl:.4f} acc {va:.4f} "
              f"| {sec:.0f}s", flush=True)
        epochs_run = ep
        if a.max_hours and time.time() - t_train > a.max_hours * 3600 and ep < a.epochs:
            print(f"[time cap] {a.max_hours} h reached after epoch {ep}: stopping", flush=True)
            break
    train_min = (time.time() - t_train) / 60
    log.close()
    torch.save(model.state_dict(), out / "last.pt")

    y, p_last = predict(model, loaders["test"], device)          # last epoch, like the authors
    model.load_state_dict(torch.load(out / "best.pt", map_location=device))
    y, p_best = predict(model, loaders["test"], device)          # best-validation checkpoint
    m_best, m_last = metrics(y, p_best, classes), metrics(y, p_last, classes)
    (out / "test_report.txt").write_text(m_best["report"])

    pc = lambda m: {"acc_%": round(100 * m["accuracy"], 2), "P_%": round(100 * m["precision_macro"], 2),
                    "R_%": round(100 * m["recall_macro"], 2), "F1_%": round(100 * m["f1_macro"], 2)}
    res = {
        "paper": "#8 Afrin et al. 2023, CNN and GFP-GAN (arXiv:2312.10701)",
        "run": out.name, "kind": "char-classifier", "note": a.note,
        "classes": classes, "n_classes": len(classes),
        "counts": {k: len(v) for k, v in items.items()},
        "settings": {"task": a.task, "model": a.model, "imgsz": a.imgsz, "epochs": a.epochs, "batch": a.batch,
                     "lr": a.lr, "optimizer": "Adam (eps 1e-7, Keras default)", "loss": "cross-entropy",
                     "augment": "none" if a.no_aug else str(aug), "imagenet_norm": a.imagenet_norm,
                     "resize": "PIL bilinear", "seed": a.seed, "data": a.data, "max_hours": a.max_hours},
        "params_total": n_params, "params_trainable": n_train,
        "epochs_planned": a.epochs, "epochs_run": epochs_run,
        "best_epoch": best_epoch, "val_acc_best": best_acc, "chars_val_best_%": round(100 * best_acc, 2),
        # in % for the results tables: 'last' = the model after the last epoch (what the paper reports),
        # 'best' = the checkpoint with the best validation accuracy (usual practice); P/R/F1 are macro averages
        "chars_test": {"last": pc(m_last), "best": pc(m_best)},
        "test_best": {k: v for k, v in m_best.items() if k != "report"},
        "test_last_epoch": {k: v for k, v in m_last.items() if k not in ("report", "confusion")},
        "train_minutes": round(train_min, 2),
        "train_device": gpu_name(device),
        "torch": torch.__version__,
    }
    (out / "results.json").write_text(json.dumps(res, indent=2))
    print(f"[{out.name}] DONE best epoch {best_epoch} val {best_acc:.4f} | TEST acc {m_best['accuracy']:.4f} "
          f"macro-F1 {m_best['f1_macro']:.4f} (last epoch acc {m_last['accuracy']:.4f}) | {train_min:.1f} min",
          flush=True)


if __name__ == "__main__":
    main()

Writing /kaggle/working/lprkit/p08_train.py


In [11]:
%%writefile /kaggle/working/lprkit/p08_chars.py
"""Paper #8 on our data: cut every labelled Bangla token out of the 498R recognition crops, so the paper's
per-symbol classifiers (CNN, ResNet-50) can be trained on them.

  python p08_chars.py --out /tmp/p08_chars [--max-images 300]

recognition/{images,labels}/{train,val,test} (YOLO boxes, 101 token classes: digits, মেট্রো, series letters, district
words) -> <out>/{training_data,val_data,testing_data}/cNNN/<crop>_<k>.png, the same folder layout as the authors'
Data.zip (one folder per class), so p08_train.py --task paper reads it unchanged. Folder cNNN = class id NNN of
recognition/data.yaml; tokens.json maps it back to the Bangla token. Train uses all train crops (human + verified +
auto boxes, like the other kits' REC data); val/test are human labels only. Each box gets 4 % padding; boxes under
4 px are skipped. The split is the dataset's grouped split, so no plate appears in two splits.
"""
import argparse
import json
import os
import sys
from collections import Counter
from pathlib import Path

import yaml
from PIL import Image

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import common

SPLITS = {'train': 'training_data', 'val': 'val_data', 'test': 'testing_data'}


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--out', required=True)
    p.add_argument('--pad', type=float, default=0.04)
    p.add_argument('--max-images', type=int, default=None, help='quick test: only N crops per split')
    a = p.parse_args()
    root = Path(common.find_dataset()) / 'recognition'
    names = yaml.safe_load(open(root / 'data.yaml', encoding='utf-8'))['names']
    names = [names[i] for i in sorted(names)] if isinstance(names, dict) else list(names)
    out = Path(a.out)
    counts, plates = {}, {}
    for split, sub in SPLITS.items():
        c = Counter()
        imgs = sorted((root / 'images' / split).iterdir())[:a.max_images]
        for f in imgs:
            lab = root / 'labels' / split / (f.stem + '.txt')
            if not lab.exists():
                continue
            im = Image.open(f).convert('RGB')
            W, H = im.size
            for k, line in enumerate(l for l in lab.read_text().splitlines() if l.strip()):
                cls, cx, cy, w, h = line.split()[:5]
                cls, cx, cy, w, h = int(cls), float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
                if w < 4 or h < 4:
                    continue
                pw, ph = a.pad * w, a.pad * h
                box = (max(0, int(cx - w / 2 - pw)), max(0, int(cy - h / 2 - ph)),
                       min(W, int(round(cx + w / 2 + pw))), min(H, int(round(cy + h / 2 + ph))))
                d = out / sub / f'c{cls:03d}'
                d.mkdir(parents=True, exist_ok=True)
                im.crop(box).save(d / f'{f.stem}_{k}.png')
                c[cls] += 1
        counts[split] = c
        plates[split] = len(imgs)
        print(f'{split}: {sum(c.values())} token crops from {len(imgs)} plate crops, {len(c)} classes', flush=True)
    tokens = {f'c{i:03d}': n for i, n in enumerate(names)}
    rare = sorted((tokens[f'c{k:03d}'], v) for k, v in counts['train'].items() if v < 10)
    info = {'tokens': tokens, 'plates': plates,
            'crops': {s: sum(c.values()) for s, c in counts.items()},
            'per_class': {tokens[f'c{k:03d}']: [counts[s].get(k, 0) for s in SPLITS] for k in sorted(set().union(*counts.values()))},
            'test_classes_without_train_crops': [tokens[f'c{k:03d}'] for k in counts['test'] if k not in counts['train']],
            'rare_train_classes(<10)': rare}
    common.save_json(info, out / 'tokens.json')
    print('classes with test crops but no train crop:', info['test_classes_without_train_crops'])
    print('saved', out / 'tokens.json')


if __name__ == '__main__':
    main()

Writing /kaggle/working/lprkit/p08_chars.py


In [12]:
%%writefile /kaggle/working/lprkit/p08_plates.py
"""Paper #8 on our data: read whole plates with the paper's pipeline and score them like the paper and like the kit.

  python p08_plates.py --cnn runs/cnn28 --name plates_cnn28 --project runs [--max 40]

Pipeline (the authors' Code/GFPGAN.ipynb + Image_Processing.ipynb + CNN.ipynb, as in the original-paper check):
  enhance -> grey -> top-hat/black-hat contrast -> 333x335 -> Otsu -> erode/dilate -> 15 largest contours
  -> order -> each blob to 28x28 -> the CNN trained on our token crops (p08_train.py) -> tokens -> text
enhance  none   : the crop as it is
         gfpgan : GFPGAN v1.3 as the authors ran it (-s 2, Real-ESRGAN background upsampler). A plate has no face, so the
                  output IS the RealESRGAN_x2plus upsampling; we run that network directly (spandrel)
order    paper  : all boxes left to right (the authors' code; it mixes the two plate lines)
         fixed  : drop too big / too small boxes, then top line left to right, then bottom line
gt_boxes        : the same CNN on the HUMAN token boxes of recognition/test (perfect segmentation), joined with the
                  kit's BRTA decoder (vocab.plate_text): the ceiling of a per-symbol classifier on our plates
On ocr/test.csv (human plate text). Scores: mean difflib SequenceMatcher similarity and the share >= 0.95 / >= 0.90
(paper p.5), plate exact / CER / char accuracy (metrics.score_rows), ms per plate for the enhancement and for
segmentation + CNN. ocr_test.rows_decoder = gfpgan + fixed order (for the team tables).
"""
import argparse
import csv
import difflib
import json
import os
import sys
import tempfile
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import yaml

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import common
import metrics
import vocab
from p08_train import PaperCNN

ESRGAN_URL = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth'


def load_upsampler(device):
    if os.environ.get('LPR_LOCAL_TEST'):
        print('local test: bicubic x2 stands in for Real-ESRGAN x2 (third-party weights are only loaded on Kaggle)')
        return None
    from spandrel import ModelLoader
    path = Path(tempfile.gettempdir()) / 'RealESRGAN_x2plus.pth'   # not in the output folder
    if not path.exists():
        torch.hub.download_url_to_file(ESRGAN_URL, str(path))
    model = ModelLoader().load_from_file(str(path)).model.to(device).eval()
    return model.half() if device.type == 'cuda' else model


@torch.no_grad()
def upsample_x2(model, bgr, device):
    if model is None:
        return cv2.resize(bgr, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    h, w = bgr.shape[:2]
    img = cv2.copyMakeBorder(bgr, 0, (-h) % 4, 0, (-w) % 4, cv2.BORDER_REFLECT)
    x = torch.from_numpy(img[:, :, ::-1].copy()).permute(2, 0, 1)[None].float().div(255).to(device)
    x = x.half() if device.type == 'cuda' else x
    y = model(x).float().clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy()
    return (y[: 2 * h, : 2 * w, ::-1] * 255).round().astype(np.uint8)


def maximise_contrast(bgr):
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    return cv2.subtract(cv2.add(gray, cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, k)),
                        cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, k))


def segment(gray, order):
    """-> list of (blob 24x44 white-bordered, box x, y, w, h) in reading order (the authors' Image_Processing code)."""
    lp = cv2.resize(gray, (333, 335))
    _, binary = cv2.threshold(lp, 200, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    binary = cv2.erode(binary, (3, 3))      # the authors pass a tuple as the kernel; kept as-is
    binary = cv2.dilate(binary, (3, 3))
    binary[0:3, :] = 255
    binary[:, 0:3] = 255
    binary[332:335, :] = 255
    binary[:, 330:333] = 255
    cntrs, _ = cv2.findContours(binary.copy(), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    boxes = [cv2.boundingRect(c) for c in sorted(cntrs, key=cv2.contourArea, reverse=True)[:15]]
    if order == 'fixed':
        H, W = binary.shape
        boxes = [b for b in boxes if b[2] < 0.6 * W and b[3] < 0.6 * H and b[2] * b[3] > 0.002 * W * H]
        if boxes:
            cy = sorted(b[1] + b[3] / 2 for b in boxes)
            gaps = [(cy[i + 1] - cy[i], i) for i in range(len(cy) - 1)]
            gap, i = max(gaps) if gaps else (0, 0)
            cut = (cy[i] + cy[i + 1]) / 2 if gap > 0.12 * H else H
            boxes = (sorted([b for b in boxes if b[1] + b[3] / 2 < cut], key=lambda b: b[0]) +
                     sorted([b for b in boxes if b[1] + b[3] / 2 >= cut], key=lambda b: b[0]))
    else:
        boxes = sorted(boxes, key=lambda b: b[0])
    out = []
    for x, y, w, h in boxes:
        canvas = np.full((44, 24), 255, np.uint8)   # 24x44 with a white border, as in the authors' code
        canvas[2:42, 2:22] = cv2.resize(binary[y:y + h, x:x + w], (20, 40))
        out.append((canvas, (x, y, w, h)))
    return out


@torch.no_grad()
def classify(model, imgs, device, imgsz):
    """imgs: grey (the authors' binarised blobs) or BGR crops -> class ids. Input as in training: RGB, /255."""
    if not imgs:
        return []
    x = []
    for im in imgs:
        im = cv2.cvtColor(im, cv2.COLOR_GRAY2RGB) if im.ndim == 2 else cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
        x.append(cv2.resize(im, (imgsz, imgsz), interpolation=cv2.INTER_AREA))
    x = torch.from_numpy(np.stack(x).astype(np.float32) / 255.0).permute(0, 3, 1, 2).to(device)
    return model(x).argmax(1).tolist()


def sim(p, t):
    return difflib.SequenceMatcher(None, vocab.norm_text(t), vocab.norm_text(p or '')).ratio()


def summary(rows):
    s = np.array([sim(p, t) for p, t in rows]) if rows else np.zeros(1)
    return dict(metrics.score_rows(rows), **{'similarity_%': round(float(100 * s.mean()), 2),
                                            'ge95_%': round(float(100 * (s >= 0.95).mean()), 2),
                                            'ge90_%': round(float(100 * (s >= 0.90).mean()), 2)})


def sync(device):
    if device.type == 'cuda':
        torch.cuda.synchronize()


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--cnn', required=True, help='run folder of a p08_train.py CNN trained on our token crops')
    p.add_argument('--name', required=True)
    p.add_argument('--project', required=True)
    p.add_argument('--max', type=int, default=None)
    a = p.parse_args()
    run = Path(a.project) / a.name
    run.mkdir(parents=True, exist_ok=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    r = json.loads((Path(a.cnn) / 'results.json').read_text())
    imgsz = r['settings']['imgsz']
    model = PaperCNN(len(r['classes']), imgsz).to(device).eval()
    model.load_state_dict(torch.load(Path(a.cnn) / 'best.pt', map_location=device))
    root = Path(common.find_dataset())
    names = yaml.safe_load(open(root / 'recognition' / 'data.yaml', encoding='utf-8'))['names']
    token = [names[int(c[1:])] for c in r['classes']]           # folder cNNN -> Bangla token
    rows = list(csv.DictReader(open(root / 'ocr' / 'test.csv', encoding='utf-8')))[:a.max]
    up = load_upsampler(device)

    res, per, ms_enh, ms_read = {}, [], [], []
    keep = {}
    for enh in ('none', 'gfpgan'):
        imgs = {}
        for x in rows:
            bgr = cv2.imread(str(root / 'ocr' / 'images' / 'test' / x['file']))
            if enh == 'gfpgan':
                sync(device); t = time.perf_counter()
                bgr = upsample_x2(up, bgr, device)
                sync(device); ms_enh.append(1000 * (time.perf_counter() - t))
                if len(ms_enh) <= 8:
                    cv2.imwrite(str(run / f'enh_{Path(x["file"]).stem}.png'), bgr)
            imgs[x['file']] = maximise_contrast(bgr)
        for order in ('paper', 'fixed'):
            key = f'{enh}_{order}'
            preds = []
            for x in rows:
                t = time.perf_counter()
                blobs = segment(imgs[x['file']], order)
                toks = [token[i] for i in classify(model, [b[0] for b in blobs], device, imgsz)]
                sync(device); ms_read.append(1000 * (time.perf_counter() - t))
                pred = ''.join(toks)            # tokens in the order the pipeline found them (the paper's string)
                preds.append({'file': x['file'], 'true': x['text'], 'pred': pred, 'source': x.get('source', '')})
                per.append({'file': x['file'], 'setting': key, 'true': x['text'], 'pred': pred,
                            'similarity': round(sim(pred, x['text']), 3)})
            res[key] = summary([(q['pred'], q['true']) for q in preds])
            keep[key] = preds
            print(key, res[key], flush=True)

    # perfect segmentation: the human token boxes of recognition/test, read by the same CNN
    rec_imgs = sorted((root / 'recognition' / 'images' / 'test').iterdir())[:a.max]
    gt_rows = []
    for f in rec_imgs:
        lab = root / 'recognition' / 'labels' / 'test' / (f.stem + '.txt')
        if not lab.exists():
            continue
        bgr = cv2.imread(str(f))
        H, W = bgr.shape[:2]
        boxes, truth = [], []
        for line in (l for l in lab.read_text().splitlines() if l.strip()):
            c, cx, cy, w, h = line.split()[:5]
            cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
            b = [max(0, int(cx - w / 2)), max(0, int(cy - h / 2)), min(W, int(round(cx + w / 2))), min(H, int(round(cy + h / 2)))]
            if b[2] - b[0] >= 2 and b[3] - b[1] >= 2:
                boxes.append(b)
                truth.append([names[int(c)]] + b)
        ids = classify(model, [bgr[b[1]:b[3], b[0]:b[2]] for b in boxes], device, imgsz)
        pred, _ = vocab.plate_text([[token[i]] + b for i, b in zip(ids, boxes)])
        gt_rows.append((pred, vocab.plate_text(truth)[0]))
    res['gt_boxes'] = dict(summary(gt_rows), crops=len(gt_rows))
    print('gt_boxes', res['gt_boxes'], flush=True)

    best = keep['gfpgan_fixed']
    out = {'run': a.name, 'kind': 'paper08-pipeline', 'cnn': str(a.cnn), 'n_classes': len(token),
           'settings': {'model': f"paper CNN {imgsz}x{imgsz} ({Path(a.cnn).name})",
                        'enhance': 'GFPGAN v1.3 -s 2 = RealESRGAN_x2plus' + (' (bicubic stand-in, local test)' if up is None else ''),
                        'segmentation': 'Otsu + 15 largest contours (authors\' code)'},
           'plates': res, 'ms_enhance': round(float(np.mean(ms_enh[3:] or ms_enh or [0])), 1),
           'ms_read': round(float(np.mean(ms_read)), 1),
           'ocr_test': {'rows_decoder': summary([(q['pred'], q['true']) for q in best]),
                        'ms_per_plate': round(float(np.mean(ms_enh[3:] or ms_enh or [0]) + np.mean(ms_read)), 1),
                        'by_condition': metrics.condition_scores(best, str(root)),
                        'note': 'rows_decoder = GFP-GAN + fixed (two-line) order'},
           'device': torch.cuda.get_device_name(0) if device.type == 'cuda' else 'cpu', 'env': common.env_info()}
    with open(run / 'per_plate.csv', 'w', newline='', encoding='utf-8') as fh:
        w = csv.DictWriter(fh, fieldnames=list(per[0]))
        w.writeheader(); w.writerows(per)
    common.save_json(out, run / 'results.json')
    print('DONE', a.name)


if __name__ == '__main__':
    main()

Writing /kaggle/working/lprkit/p08_plates.py


## 2. Dataset check

In [13]:
# ===== 2. Dataset ====================================================================================================
import common
ROOT = common.find_dataset(); print("dataset:", ROOT)   # attached Kaggle input, or auto-download from the team Drive
os.environ["DATASET_ROOT"] = ROOT                        # every job below uses this copy (no second download)
MAN = common.load_json(f"{ROOT}/dataset_manifest.json")
for k, v in MAN["counts"].items(): print(f"  {k:40s} {v}")
DET   = common.data_yaml("detection", "all")      # human + auto labels in train; val/test human only
DET_H = common.data_yaml("detection", "human")    # human labels only
REC   = common.data_yaml("recognition", "all")
REC_H = common.data_yaml("recognition", "human")
print(open(DET).read())
common.save_json(dict(common.env_info(), dataset_version=MAN.get("version", "?")), f"{WORK}/environment.json")


# ---- job helpers: each returns one command line; RUN starts them (one queue per GPU) and shows failures
PY = sys.executable
OCRMAX = ["--ocr-max", "40"] if FAST else []
SEQMAX = ["--max-test", "40"] if FAST else []
def DETJOB(name, model, epochs, data=None, imgsz=640, batch=16, hours=None, extra=()):
    return [PY, f"{KIT}/det.py", "--data", data or DET, "--model", model, "--name", name, "--project", RUNS,
            "--epochs", E(epochs), "--imgsz", imgsz, "--batch", batch, "--fraction", FR, *TMAX,
            *(["--max-hours", hours] if hours and not FAST else []), *extra]
def RECJOB(name, model, epochs, data=None, imgsz=320, batch=32, hours=None, extra=()):
    return [PY, f"{KIT}/rec.py", "--data", data or REC, "--model", model, "--name", name, "--project", RUNS,
            "--epochs", E(epochs), "--imgsz", imgsz, "--batch", batch, "--fraction", FR, *OCRMAX,
            *(["--max-hours", hours] if hours and not FAST else []), *extra]
def SEQJOB(name, model, epochs, extra=()):
    return [PY, f"{KIT}/seq.py", "--model", model, "--name", name, "--project", RUNS, "--epochs", E(epochs),
            *SEQMAX, *(["--real-fraction", "0.1"] if FAST else []), *extra]
def VEDJOB(name, epochs, extra=()):
    return [PY, f"{KIT}/ved.py", "--name", name, "--project", RUNS, "--epochs", E(epochs),
            *(["--max-train", "200", "--max-test", "20"] if FAST else []), *extra]
def SPEED(name, data=None, imgsz=640, trt=None, rtdetr=False):
    """batch-1 latency of a trained detector: PyTorch FP32/FP16 (+ TensorRT FP16 / INT8) on the T4, ONNX on the CPU."""
    t = [] if (FAST or not trt) else (["--trt"] if trt == "fp16" else ["--trt", trt])
    return [PY, f"{KIT}/det.py", "--data", data or DET, "--name", name, "--project", RUNS, "--imgsz", imgsz,
            "--fps-only", *t, *(["--rtdetr"] if rtdetr else [])]
def X(script, *args):
    return [PY, f"{KIT}/{script}", *args]
def RUN(jobs):
    """jobs: (name, gpu, cmd[, env]). Same gpu -> one after the other; different gpus -> at the same time."""
    jobs = [(j[0], j[1], [str(c) for c in j[2]]) + tuple(j[3:]) for j in jobs]
    codes = common.run_on_gpus(jobs, f"{WORK}/logs")
    for n, c in codes.items():
        if c != 0:
            print(f"----- {n} failed (code {c}); last lines of logs/{n}.log:")
            print(common.tail(f"{WORK}/logs/{n}.log", 25))
    return codes

dataset not attached -> downloading it from the team Drive (https://drive.google.com/drive/folders/11psdzW5Cpr7tXwqzqNxNt3LA-qeDSOQl)
  0.00 / 3.38 GB
  0.49 / 3.38 GB
  0.98 / 3.38 GB
  1.46 / 3.38 GB
  1.95 / 3.38 GB
  2.44 / 3.38 GB
  2.93 / 3.38 GB
  unpacking ...
dataset ready: /tmp/498r_data/BanglaLPR_AllInOne_v1_1
dataset: /tmp/498r_data/BanglaLPR_AllInOne_v1_1
  det|test|human                           1153
  det|train|auto                           4507
  det|train|human                          4067
  det|val|human                            896
  e2e|test                                 204
  extras_fog|test                          26
  extras_fog|train                         267
  extras_fog|val                           59
  leaky_kuet|detection|fair_test           540
  leaky_kuet|detection|fair_train          1758
  leaky_kuet|detection|fair_val            311
  leaky_kuet|detection|test|copy           27
  leaky_kuet|detection|test|unique         526
  leaky_kuet|dete

## 3. Token crops from our recognition split (one folder per Bangla token)

In [14]:
CH = "/tmp/p08_chars"                      # outside /kaggle/working: not saved as output
r = subprocess.run([PY, f"{KIT}/p08_chars.py", "--out", CH, *(["--max-images", "300"] if FAST else [])],
                   capture_output=True, encoding="utf-8", errors="replace")
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0 and os.path.exists(f"{CH}/tokens.json"), "the token crops were not made: see the error above"

train: 47992 token crops from 5465 plate crops, 99 classes
val: 4643 token crops from 536 plate crops, 85 classes
test: 4107 token crops from 467 plate crops, 60 classes
classes with test crops but no train crop: ['মুন্সিগঞ্জ']
saved /tmp/p08_chars/tokens.json
 


## 4. Train the paper's CNN (GPU 0) and ResNet-50 (GPU 1) on our token crops

In [15]:
try:
    LIM = ["--limit", "300"] if FAST else []
    P8 = lambda name, model, epochs, *extra: X("p08_train.py", "--task", "paper", "--model", model, "--data", CH,
                                              "--out", f"{RUNS}/{name}", "--epochs", E(epochs), *LIM, *extra)
    codes = RUN([
        ("cnn28", G0, P8("cnn28", "cnn", 40, "--max-hours", 3, "--note", "paper #8 CNN 28x28, batch 1, Adam 1e-4 (authors' code)")),
        ("resnet50", G1, P8("resnet50", "resnet50", 30, "--max-hours", 3.2, "--note", "frozen ResNet-50 150x150, batch 1 (authors' code)")),
    ])
except Exception:          # keep going: the results table and the outputs zip must still be written
    import traceback; traceback.print_exc()

[start] cnn28 on GPU 0: /usr/bin/python3 /kaggle/working/lprkit/p08_train.py --task paper --model cnn --data /tmp/p08_chars --out /kaggle/working/runs/cnn28 --epochs 40 --max-hours 3 --note paper #8 CNN 28x28, batch 1, Adam 1e-4 (authors' code)
[start] resnet50 on GPU 1: /usr/bin/python3 /kaggle/working/lprkit/p08_train.py --task paper --model resnet50 --data /tmp/p08_chars --out /kaggle/working/runs/resnet50 --epochs 30 --max-hours 3.2 --note frozen ResNet-50 150x150, batch 1 (authors' code)
[done]  cnn28 code=0 after 152.5 min
[done]  resnet50 code=0 after 199.5 min


## 5. The paper's plate pipeline, with and without GFP-GAN (GPU 0), and classifier speed (GPU 1)

In [16]:
try:
    MAXP = ["--max", 40] if FAST else []
    RUN([("plates_cnn28", G0, X("p08_plates.py", "--cnn", f"{RUNS}/cnn28", "--name", "plates_cnn28", "--project", RUNS, *MAXP)),
         ("fps_cnn28", G1, X("p08_train.py", "--fps-only", "--out", f"{RUNS}/cnn28")),
         ("fps_resnet50", G1, X("p08_train.py", "--fps-only", "--out", f"{RUNS}/resnet50"))])
    for r in ("cnn28", "resnet50"):
        d = common.load_json(f"{RUNS}/{r}/results.json"); print(r, d["n_classes"], "classes |", d["chars_test"], d.get("speed"))
    print(json.dumps(common.load_json(f"{RUNS}/plates_cnn28/results.json")["plates"], indent=1, ensure_ascii=False))
except Exception:          # keep going: the results table and the outputs zip must still be written
    import traceback; traceback.print_exc()

[start] plates_cnn28 on GPU 0: /usr/bin/python3 /kaggle/working/lprkit/p08_plates.py --cnn /kaggle/working/runs/cnn28 --name plates_cnn28 --project /kaggle/working/runs
[start] fps_cnn28 on GPU 1: /usr/bin/python3 /kaggle/working/lprkit/p08_train.py --fps-only --out /kaggle/working/runs/cnn28
[done]  fps_cnn28 code=0 after 0.2 min
[start] fps_resnet50 on GPU 1: /usr/bin/python3 /kaggle/working/lprkit/p08_train.py --fps-only --out /kaggle/working/runs/resnet50
[done]  fps_resnet50 code=0 after 0.2 min
[done]  plates_cnn28 code=0 after 3.0 min
cnn28 101 classes | {'last': {'acc_%': 36.67, 'P_%': 13.69, 'R_%': 7.15, 'F1_%': 8.16}, 'best': {'acc_%': 93.08, 'P_%': 38.65, 'R_%': 36.16, 'F1_%': 36.42}} {'fps_bs1': 760.9, 'ms_per_image': 1.31, 'device': 'Tesla T4', 'note': 'model only, batch 1, FP32, measured alone on an idle GPU'}
resnet50 101 classes | {'last': {'acc_%': 92.06, 'P_%': 49.03, 'R_%': 49.02, 'F1_%': 46.41}, 'best': {'acc_%': 92.89, 'P_%': 47.79, 'R_%': 45.08, 'F1_%': 43.9}} {'f

## Results vs. the paper, and the outputs zip

In [17]:
import report, shutil
rows = [('Classes', '17 (10 digits, 4 letters, 3 region words)', 'p.3, Table I', 'n_classes'), ('Character test accuracy, last epoch (%)', '92.38 (CNN), 87.50 (ResNet-50)', 'p.4, Table II', 'chars_test.last.acc_%'), ('Character macro precision (%)', '96.67 (CNN), 93.43 (ResNet-50)', 'p.4, Table II', 'chars_test.last.P_%'), ('Character macro recall (%)', '91.62 (CNN), 88.27 (ResNet-50)', 'p.4, Table II', 'chars_test.last.R_%'), ('Character macro F1 (%)', '94.76 (CNN), 89.84 (ResNet-50)', 'p.4, Table II', 'chars_test.last.F1_%'), ('Character test accuracy, best-val checkpoint (%)', '-', '', 'chars_test.best.acc_%'), ('Best validation accuracy (%)', '98.80 (CNN)', 'p.4', 'chars_val_best_%'), ('Epochs run', '40 (CNN), 30 (ResNet-50)', 'p.4-5', 'epochs_run'), ('Classifier FPS, batch 1, T4 (model only)', 'not reported', '', 'speed.fps_bs1'), ('Training time (min)', 'not reported', '', 'train_minutes'), ('Plates: mean similarity, GFP-GAN + x order (%)', '88 (30 plates)', 'p.5', 'plates.gfpgan_paper.similarity_%'), ('Plates with similarity >= 0.95 (%)', '57 (17 of 30)', 'p.5', 'plates.gfpgan_paper.ge95_%'), ('Plates with similarity >= 0.90 (%)', '80 (24 of 30)', 'p.5', 'plates.gfpgan_paper.ge90_%'), ('Plates: mean similarity WITHOUT GFP-GAN (%)', 'never tested', '', 'plates.none_paper.similarity_%'), ('Plates exact, GFP-GAN + two-line order (%)', 'not reported', '', 'plates.gfpgan_fixed.plate_exact_%'), ('Plates exact, no GFP-GAN + two-line order (%)', '-', '', 'plates.none_fixed.plate_exact_%'), ('Plates exact, perfect segmentation (human token boxes) (%)', '-', '', 'plates.gt_boxes.plate_exact_%'), ('Char accuracy, perfect segmentation (%)', '-', '', 'plates.gt_boxes.char_acc_%'), ('GFP-GAN (Real-ESRGAN x2) ms per plate', 'not reported', '', 'ms_enhance'), ('Segmentation + CNN ms per plate', 'not reported', '', 'ms_read')]
md = report.compare_table(RUNS, rows, ['cnn28', 'resnet50', 'plates_cnn28'], "Paper #8: ours vs. the paper", "The paper's 17 classes and 303 test crops (48 are copies of training crops, see the original check) vs our 101 tokens on the grouped split: a harder task, so a lower symbol accuracy is expected. The paper's plate score is the mean difflib SequenceMatcher ratio, which gives partial credit; plate exact is the strict number.")
md += report.condition_table(RUNS, sorted(d for d in os.listdir(RUNS) if os.path.isdir(f"{RUNS}/{d}")))
open(f"{WORK}/results_vs_paper.md", "w", encoding="utf-8").write(md); print(md)
os.makedirs(f"{WORK}/code", exist_ok=True)
for f in glob.glob(f"{KIT}/*.py"): shutil.copy(f, f"{WORK}/code/")
report.pack(WORK, "paper08")

# Paper #8: ours vs. the paper

| Metric | Paper (page) | cnn28 | resnet50 | plates_cnn28 |
|---|---|---|---|---|
| Classes | 17 (10 digits, 4 letters, 3 region words) (p.3, Table I) | 101 | 101 | 101 |
| Character test accuracy, last epoch (%) | 92.38 (CNN), 87.50 (ResNet-50) (p.4, Table II) | 36.67 | 92.06 | – |
| Character macro precision (%) | 96.67 (CNN), 93.43 (ResNet-50) (p.4, Table II) | 13.69 | 49.03 | – |
| Character macro recall (%) | 91.62 (CNN), 88.27 (ResNet-50) (p.4, Table II) | 7.15 | 49.02 | – |
| Character macro F1 (%) | 94.76 (CNN), 89.84 (ResNet-50) (p.4, Table II) | 8.16 | 46.41 | – |
| Character test accuracy, best-val checkpoint (%) | - | 93.08 | 92.89 | – |
| Best validation accuracy (%) | 98.80 (CNN) (p.4) | 90.20 | 91.71 | – |
| Epochs run | 40 (CNN), 30 (ResNet-50) (p.4-5) | 40 | 25 | – |
| Classifier FPS, batch 1, T4 (model only) | not reported | 760.90 | 171.50 | – |
| Training time (min) | not reported | 151.80 | 198.20 | – |
| Plates: mean similarity, GFP

'/kaggle/working/paper08_outputs.zip'

Download `paper08_outputs.zip` from the **Output** tab and upload it to the team Drive (see the guide).